# CareGap AI — CKD Guideline RAG Experiment

## Objective

Build a small clinical AI experiment that:

1. Loads synthetic patient records containing structured and unstructured information.
2. Loads a public CKD clinical guideline.
3. Retrieves relevant guideline evidence using semantic search.
4. Uses an LLM to reason over the patient record and retrieved evidence.
5. Produces a structured clinical finding using Pydantic.
6. Preserves evidence and provenance for both the patient data and guideline.

### Important

This is a research/portfolio prototype using synthetic data.

It is **not a clinical decision-support system** and must not be used for real patient care.

## Architecture

The first version intentionally avoids agents.

```text
Synthetic Patient
       |
       v
Patient Facts
       |
       +----------------------+
       |                      |
       v                      v
Structured Data        Clinical Notes
       |                      |
       +----------+-----------+
                  |
                  v
            Patient Context
                  |
                  v
          Guideline Retriever
                  |
                  v
        Relevant Guideline Evidence
                  |
                  v
             LLM Reasoning
                  |
                  v
          Pydantic Validation
                  |
                  v
       Clinical Finding + Evidence

In [2]:
import os
import json
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI

from pydantic import BaseModel, Field

from pypdf import PdfReader

from sklearn.metrics.pairwise import cosine_similarity

In [3]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY not found. Add it to your .env file."
    )

client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client initialised.")

OpenAI client initialised.


## 1. Create synthetic patient data

We will deliberately separate:

- demographics
- diagnoses
- medications
- laboratory results
- encounters
- clinical notes

This gives us both **structured and unstructured patient information**.

In [4]:
patients = [
    {
        "patient_id": "P001",
        "demographics": {
            "age": 67,
            "sex": "M"
        },
        "diagnoses": [
            "Hypertension"
        ],
        "medications": [
            "Amlodipine 10 mg daily"
        ],
        "labs": [
            {
                "date": "2026-04-12",
                "test": "eGFR",
                "value": 48,
                "unit": "mL/min/1.73m2"
            },
            {
                "date": "2026-04-12",
                "test": "Creatinine",
                "value": 1.6,
                "unit": "mg/dL"
            }
        ],
        "encounters": [
            {
                "date": "2026-04-12",
                "type": "GP consultation"
            }
        ],
        "clinical_notes": [
            {
                "date": "2026-04-12",
                "text": (
                    "Patient reports increasing fatigue over recent months. "
                    "Blood pressure remains elevated despite current medication. "
                    "Recent blood testing demonstrated reduced kidney function."
                )
            }
        ]
    },

    {
        "patient_id": "P002",
        "demographics": {
            "age": 54,
            "sex": "F"
        },
        "diagnoses": [
            "Hypertension",
            "Chronic kidney disease"
        ],
        "medications": [
            "Ramipril 10 mg daily"
        ],
        "labs": [
            {
                "date": "2026-03-10",
                "test": "eGFR",
                "value": 38,
                "unit": "mL/min/1.73m2"
            },
            {
                "date": "2026-03-10",
                "test": "Creatinine",
                "value": 1.8,
                "unit": "mg/dL"
            }
        ],
        "encounters": [
            {
                "date": "2026-03-10",
                "type": "GP consultation"
            }
        ],
        "clinical_notes": [
            {
                "date": "2026-03-10",
                "text": (
                    "Known chronic kidney disease. Patient reports no new symptoms. "
                    "Kidney function remains reduced on repeat testing."
                )
            }
        ]
    },

    {
        "patient_id": "P003",
        "demographics": {
            "age": 42,
            "sex": "M"
        },
        "diagnoses": [],
        "medications": [],
        "labs": [
            {
                "date": "2026-05-02",
                "test": "eGFR",
                "value": 94,
                "unit": "mL/min/1.73m2"
            },
            {
                "date": "2026-05-02",
                "test": "Creatinine",
                "value": 0.9,
                "unit": "mg/dL"
            }
        ],
        "encounters": [
            {
                "date": "2026-05-02",
                "type": "Routine blood test"
            }
        ],
        "clinical_notes": [
            {
                "date": "2026-05-02",
                "text": (
                    "Routine blood testing. No known kidney disease. "
                    "Patient has no current renal symptoms."
                )
            }
        ]
    }
]

print(f"Loaded {len(patients)} synthetic patients.")

Loaded 3 synthetic patients.


In [5]:
print(json.dumps(patients[0], indent=2))

{
  "patient_id": "P001",
  "demographics": {
    "age": 67,
    "sex": "M"
  },
  "diagnoses": [
    "Hypertension"
  ],
  "medications": [
    "Amlodipine 10 mg daily"
  ],
  "labs": [
    {
      "date": "2026-04-12",
      "test": "eGFR",
      "value": 48,
      "unit": "mL/min/1.73m2"
    },
    {
      "date": "2026-04-12",
      "test": "Creatinine",
      "value": 1.6,
      "unit": "mg/dL"
    }
  ],
  "encounters": [
    {
      "date": "2026-04-12",
      "type": "GP consultation"
    }
  ],
  "clinical_notes": [
    {
      "date": "2026-04-12",
      "text": "Patient reports increasing fatigue over recent months. Blood pressure remains elevated despite current medication. Recent blood testing demonstrated reduced kidney function."
    }
  ]
}


## 2. Load the CKD guideline

Download the KDIGO 2024 CKD guideline PDF and place it in:

```text
data/
    guidelines/
        kdigo_2024_ckd.pdf

In [6]:


### Cell 9 — Code — paths

GUIDELINE_PATH = Path("data/guidelines/kdigo_2024_ckd.pdf")

if not GUIDELINE_PATH.exists():
    raise FileNotFoundError(
        f"Guideline not found at {GUIDELINE_PATH}. "
        "Download the guideline PDF and place it there."
    )

print(GUIDELINE_PATH)

data/guidelines/kdigo_2024_ckd.pdf


In [ ]:
# use pdf reader (might lose some hieracrhical context here)
reader = PdfReader(str(GUIDELINE_PATH))

print(f"Number of PDF pages: {len(reader.pages)}")

Number of PDF pages: 199


In [8]:
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()

    if text and text.strip():
        pages.append({
            "page": page_number,
            "text": text.strip()
        })

print(f"Pages containing text: {len(pages)}")

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/PHBHGE+AdvOTce3d9a73', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 37, '/FontDescriptor': IndirectObject(10718, 0, 5895729376), '/LastChar': 122, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [833, 0, 0, 291, 291, 0, 0, 229, 312, 229, 333, 531, 531, 531, 531, 531, 531, 531, 531, 531, 531, 229, 229, 0, 0, 0, 427, 0, 635, 572, 583, 677, 510, 500, 656, 666, 260, 385, 572, 489, 822, 666, 697, 552, 0, 562, 510, 520, 656, 593, 864, 0, 572, 562, 291, 0, 291, 0, 0, 0, 500, 583, 447, 572, 510, 312, 572, 562, 250, 260, 500, 250, 843, 562, 562, 583, 572, 354, 416, 343, 562, 500, 739, 489, 500, 447]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/PHBHGF+AdvMYRLIGIT', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 73, '/FontDescriptor': Indire

Pages containing text: 199


In [9]:
print(pages[0]["text"][:3000])

VOLUME 105 | ISSUE 4S | APRIL 2024
www.kidney-international.org
KDIGO 2024 Clinical Practice Guideline for the  
Evaluation and Management of Chronic Kidney Disease
SUPPLEMENT TO


## 3. Chunk the guideline

For the first experiment we will use simple page-aware chunks.

The important thing is that every chunk retains:

- document name
- page number
- text

This gives us basic provenance.

 # Simple character-based chunking for v1

In [10]:
DOCUMENT_NAME = "KDIGO 2024 CKD Guideline"

chunks = []

for page in pages:
    text = page["text"]

   
    chunk_size = 1800
    overlap = 300

    start = 0
    chunk_index = 0

    while start < len(text):
        end = start + chunk_size

        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "chunk_id": f"page_{page['page']}_chunk_{chunk_index}",
                "document": DOCUMENT_NAME,
                "page": page["page"],
                "text": chunk_text
            })

        start += chunk_size - overlap
        chunk_index += 1

print(f"Created {len(chunks)} chunks.")

Created 764 chunks.


In [11]:
pd.DataFrame(chunks).head()

,chunk_id,document,page,text
0,page_1_chunk_0,KDIGO 2024 CKD Guideline,1,VOLUME 105 | ISSUE 4S | APRIL 2024\nwww.kidney...
1,page_2_chunk_0,KDIGO 2024 CKD Guideline,2,KDIGO 2024 CLINICAL PRACTICE GUIDELINE\nFOR TH...
2,page_3_chunk_0,KDIGO 2024 CKD Guideline,3,KDIGO 2024 Clinical Practice Guideline for the...
3,page_3_chunk_1,KDIGO 2024 CKD Guideline,3,t by the authors may re ﬂect their clinical ex...
4,page_4_chunk_0,KDIGO 2024 CKD Guideline,4,TABLES\nS137 Table 1. Criteria for chronic kid...


PARSING

In [14]:
from pathlib import Path
from docling.document_converter import DocumentConverter


GUIDELINE_PATH = Path("data/guidelines/kdigo_2024_ckd.pdf")
converter = DocumentConverter()

result = converter.convert(GUIDELINE_PATH)

doc = result.document

print(type(doc))
print("Parsed successfully")

[INFO] 2026-09-23 17:53:08,459 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-23 17:53:08,462 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-23 17:53:08,463 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.2/torch/PP-OCRv6/det/PP-OCRv6_det_small.pth
[INFO] 2026-09-23 17:53:10,240 [RapidOCR] download_file.py:82: Download size: 9.77MB
[INFO] 2026-09-23 17:53:10,558 [RapidOCR] download_file.py:95: Successfully saved to: /Users/personal/Desktop/clinical-assistant/.venv/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-23 17:53:10,563 [RapidOCR] main.py:50: Using /Users/personal/Desktop/clinical-assistant/.venv/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-23 17:53:10,647 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-23 17:53:10,647 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-23 17:53:10

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/Users/personal/Desktop/clinical-assistant/.venv/lib/python3.13/site-packages/torch/nn/modules/conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1105.)
  return F.conv2d(


2026-09-23 17:54:29,570 MatchingPostProcessor WARNING  1 of 384 pdf cells matched neither a row nor a column band of the 35x9 grid and were dropped from the table
2026-09-23 17:57:13,774 MatchingPostProcessor WARNING  7 of 331 pdf cells matched neither a row nor a column band of the 11x7 grid and were dropped from the table


[WARNING] 2026-09-23 17:58:41,578 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
[WARNING] 2026-09-23 17:58:43,581 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-09-23 17:58:43,824 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
RapidOCR returned empty result!
[WARNING] 2026-09-23 17:58:45,284 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-09-23 17:58:45,548 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-09-23 17:58:45,827 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-09-23 17:58:47,035 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty resul

<class 'docling_core.types.doc.document.DoclingDocument'>
Parsed successfully


In [15]:
print("Document type:", type(doc))

print("\nNumber of document items:")
print(len(doc.texts))

print("\nFirst 10 items:")
for i, item in enumerate(doc.texts[:10]):
    print("=" * 80)
    print("ITEM:", i)
    print("TYPE:", type(item).__name__)
    print("TEXT:", item.text[:500])

Document type: <class 'docling_core.types.doc.document.DoclingDocument'>

Number of document items:
7643

First 10 items:
ITEM: 0
TYPE: TextItem
TEXT: OFFICIAL JOURNAL OF THE INTERNATIONAL SOCIETY OF NEPHROLOGY
ITEM: 1
TYPE: TextItem
TEXT: SUPPLEMENT TO
ITEM: 2
TYPE: SectionHeaderItem
TEXT: INTERNATIONAL
ITEM: 3
TYPE: TextItem
TEXT: DISEASE
ITEM: 4
TYPE: TextItem
TEXT: KIDNEY D
ITEM: 5
TYPE: TextItem
TEXT: O
ITEM: 6
TYPE: TextItem
TEXT: R
ITEM: 7
TYPE: TextItem
TEXT: KDIGO 2024 Clinical Practice Guideline for the Evaluation and Management of Chronic Kidney Disease
ITEM: 8
TYPE: TextItem
TEXT: VOLUME 105 | ISSUE 4S | APRIL 2024
ITEM: 9
TYPE: TextItem
TEXT: www.kidney-international.org OISN


In [16]:
from collections import Counter

item_types = Counter(type(item).__name__ for item in doc.texts)

print("Item type counts:")
for item_type, count in item_types.most_common():
    print(f"{item_type}: {count}")

Item type counts:
TextItem: 5753
ListItem: 1530
SectionHeaderItem: 360


In [17]:
headers = [
    item.text.strip()
    for item in doc.texts
    if type(item).__name__ == "SectionHeaderItem"
]

print(f"Number of section headers: {len(headers)}")

for i, header in enumerate(headers[:50], start=1):
    print(f"{i:02d}. {header}")

Number of section headers: 360
01. INTERNATIONAL
02. KDIGO 2024 CLINICAL PRACTICE GUIDELINE FOR THE EVALUATION AND MANAGEMENT OF CHRONIC KIDNEY DISEASE
03. VOL 105 | ISSUE 4S | APRIL 2024
04. KDIGO 2024 Clinical Practice Guideline for the Evaluation and Management of Chronic Kidney Disease
05. TABLES
06. FIGURES
07. SUPPLEMENTARY MATERIAL
08. Appendix C. Data supplement - Summary of findings (SoF) tables cited in the guideline text
09. Chapter 1. Evaluation of CKD
10. Chapter 2. Risk assessment in people with CKD
11. Chapter 3. Delaying CKD progression and managing its complications
12. Appendix D - Data supplement - Summary of findings (SoF) tables not cited in the guideline text
13. Chapter 3. Delaying CKD progression and managing its complications
14. Appendix E - PRISMA diagrams
15. Chapter 1. Evaluation of CKD
16. Chapter 3. Delaying CKD progression and managing its complications
17. KDIGO EXECUTIVE COMMITTEE
18. KDIGO Staff
19. Reference keys
20. NOMENCLATURE AND DESCRIPTION FOR 

In [18]:
for i, item in enumerate(doc.texts):
    if item.text.strip() == "Definition and classification of CKD":
        print("Found at index:", i)

        start = max(0, i - 3)
        end = min(len(doc.texts), i + 8)

        for j in range(start, end):
            current = doc.texts[j]
            print("=" * 80)
            print("INDEX:", j)
            print("TYPE:", type(current).__name__)
            print("TEXT:", current.text[:1000])

Found at index: 1176
INDEX: 1173
TYPE: TextItem
TEXT: Kidney International (2024) 105 (Suppl 4S), S117-S314
INDEX: 1174
TYPE: TextItem
TEXT: www.kidney-international.org
INDEX: 1175
TYPE: TextItem
TEXT: introduction, qualifying statements, and key concepts
INDEX: 1176
TYPE: SectionHeaderItem
TEXT: Definition and classification of CKD
INDEX: 1177
TYPE: TextItem
TEXT: Defining CKD. CKD is defined as abnormalities of kidney structure or function, present for a minimum of 3 months, with implications for health (Table 1). 1
INDEX: 1178
TYPE: TextItem
TEXT: Classifying CKD. CKD is classified based on Cause, GFR category (G1-G5), and Albuminuria category (A1-A3), abbreviated as CGA. 1 These 3 components of the classification system are each critical in the assessment of people with CKD and help enable determination of severity and risk. Listed below are reference tables describing each component. Note that while the definition of CKD includes many different markers of kidney damage and is not

In [19]:
for i, item in enumerate(doc.texts):
    text = item.text.strip()

    if "Practice Point" in text or "Recommendation" in text:
        print("=" * 80)
        print("INDEX:", i)
        print("TYPE:", type(item).__name__)
        print("TEXT:", text[:1000])

        if i > 1500:
            break

INDEX: 284
TYPE: TextItem
TEXT: The Evidence Review Team conducts a systematic review of existing literature, extracting studies with appropriate design and outcomes deemed important by both people with CKD and clinicians. All work is meticulously graded on study quality and potential bias, forming the basis for quantifying the overall certainty of the evidence using the 'Grading of Recommendations Assessment, Development, and Evaluation' (GRADE) approach. The penultimate version of the guideline also undergoes public review to capture additional perspectives. Thus, guidelines are the result of a rigorous and objective assessment of available evidence, enriched by the collective expertise of healthcare providers, researchers, and patients alike. Guideline statements ('We recommend' or 'We suggest') reflect clinical questions that were addressed by the evidence reviews from the Evidence Review Team. Practice points provide guidance on clinical questions that were not, and largely could 

In [20]:
i = 1895

for j in range(i - 5, i + 11):
    item = doc.texts[j]

    print("=" * 80)
    print("INDEX:", j)
    print("TYPE:", type(item).__name__)
    print("TEXT:", item.text[:1200])

INDEX: 1890
TYPE: TextItem
TEXT: summary of recommendation statements and practice points
INDEX: 1891
TYPE: SectionHeaderItem
TEXT: Summary of recommendation statements and practice points
INDEX: 1892
TYPE: SectionHeaderItem
TEXT: Chapter 1: Evaluation of CKD
INDEX: 1893
TYPE: ListItem
TEXT: Detection and evaluation of CKD
INDEX: 1894
TYPE: SectionHeaderItem
TEXT: 1.1.1 Detection of CKD
INDEX: 1895
TYPE: TextItem
TEXT: Practice Point 1.1.1.1: Test people at risk for and with chronic kidney disease (CKD) using both urine albumin measurement and assessment of glomerular filtration rate (GFR).
INDEX: 1896
TYPE: TextItem
TEXT: Practice Point 1.1.1.2: Following incidental detection of elevated urinary albumin-to-creatinine ratio (ACR), hematuria, or low estimated GFR (eGFR), repeat tests to confirm presence of CKD.
INDEX: 1897
TYPE: SectionHeaderItem
TEXT: 1.1.2 Methods for staging of CKD
INDEX: 1898
TYPE: TextItem
TEXT: Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using 

In [21]:
item = doc.texts[1895]

print("TYPE:", type(item).__name__)
print("\nAvailable attributes:")
print([attr for attr in dir(item) if not attr.startswith("_")])

TYPE: TextItem

Available attributes:
['children', 'comments', 'construct', 'content_layer', 'copy', 'dict', 'export_to_doctags', 'export_to_document_tokens', 'formatting', 'from_orm', 'get_annotations', 'get_image', 'get_location_tokens', 'get_ref', 'hyperlink', 'json', 'label', 'meta', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'orig', 'parent', 'parse_file', 'parse_obj', 'parse_raw', 'prov', 'schema', 'schema_json', 'self_ref', 'source', 'text', 'update_forward_refs', 'validate']


In [22]:
item = doc.texts[1895]

print("PROVENANCE:")
print(item.prov)

PROVENANCE:
[ProvenanceItem(page_no=34, bbox=BoundingBox(l=44.4472, t=590.0392697999999, r=549.4613566000003, b=569.0573326084507, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 174))]


CHUNKING

In [23]:
from typing import Optional


chunks = []

current_chapter: Optional[str] = None
current_section: Optional[str] = None

for item in doc.texts:
    item_type = type(item).__name__
    text = item.text.strip()

    if not text:
        continue

    if item_type == "SectionHeaderItem":
        # For now, treat major "Chapter" headers as chapter boundaries.
        if text.startswith("Chapter "):
            current_chapter = text
            current_section = None

        else:
            current_section = text

        continue

    # Keep text/list content for now.
    if item_type in {"TextItem", "ListItem"}:
        chunks.append({
            "chapter": current_chapter,
            "section": current_section,
            "text": text,
            "page": item.prov[0].page_no if item.prov else None,
            "bbox": item.prov[0].bbox if item.prov else None,
        })

print(f"Created {len(chunks)} initial chunks")

for chunk in chunks[:10]:
    print("=" * 80)
    print("CHAPTER:", chunk["chapter"])
    print("SECTION:", chunk["section"])
    print("PAGE:", chunk["page"])
    print("TEXT:", chunk["text"][:500])

Created 7283 initial chunks
CHAPTER: None
SECTION: None
PAGE: 1
TEXT: OFFICIAL JOURNAL OF THE INTERNATIONAL SOCIETY OF NEPHROLOGY
CHAPTER: None
SECTION: None
PAGE: 1
TEXT: SUPPLEMENT TO
CHAPTER: None
SECTION: INTERNATIONAL
PAGE: 1
TEXT: DISEASE
CHAPTER: None
SECTION: INTERNATIONAL
PAGE: 1
TEXT: KIDNEY D
CHAPTER: None
SECTION: INTERNATIONAL
PAGE: 1
TEXT: O
CHAPTER: None
SECTION: INTERNATIONAL
PAGE: 1
TEXT: R
CHAPTER: None
SECTION: INTERNATIONAL
PAGE: 1
TEXT: KDIGO 2024 Clinical Practice Guideline for the Evaluation and Management of Chronic Kidney Disease
CHAPTER: None
SECTION: INTERNATIONAL
PAGE: 1
TEXT: VOLUME 105 | ISSUE 4S | APRIL 2024
CHAPTER: None
SECTION: INTERNATIONAL
PAGE: 1
TEXT: www.kidney-international.org OISN
CHAPTER: None
SECTION: INTERNATIONAL
PAGE: 1
TEXT: INTERNATIONAL SOCIETY


In [24]:
from collections import Counter

content_layers = Counter(
    str(getattr(item, "content_layer", None))
    for item in doc.texts
)

for layer, count in content_layers.items():
    print(count, layer)

793 ContentLayer.FURNITURE
6850 ContentLayer.BODY


# body: actual doc, furniture: small metadata

In [25]:
body_items = [
    item for item in doc.texts
    if str(item.content_layer) == "ContentLayer.BODY"
]

print(f"BODY items: {len(body_items)}")

for i, item in enumerate(body_items[:30]):
    print("=" * 80)
    print("INDEX:", i)
    print("TYPE:", type(item).__name__)
    print("TEXT:", item.text[:300])

BODY items: 6850
INDEX: 0
TYPE: TextItem
TEXT: SUPPLEMENT TO
INDEX: 1
TYPE: SectionHeaderItem
TEXT: INTERNATIONAL
INDEX: 2
TYPE: TextItem
TEXT: DISEASE
INDEX: 3
TYPE: TextItem
TEXT: KIDNEY D
INDEX: 4
TYPE: TextItem
TEXT: O
INDEX: 5
TYPE: TextItem
TEXT: R
INDEX: 6
TYPE: TextItem
TEXT: KDIGO 2024 Clinical Practice Guideline for the Evaluation and Management of Chronic Kidney Disease
INDEX: 7
TYPE: TextItem
TEXT: VOLUME 105 | ISSUE 4S | APRIL 2024
INDEX: 8
TYPE: TextItem
TEXT: www.kidney-international.org OISN
INDEX: 9
TYPE: TextItem
TEXT: INTERNATIONAL SOCIETY
INDEX: 10
TYPE: TextItem
TEXT: OF NEPHROLOGY
INDEX: 11
TYPE: TextItem
TEXT: KIDNEY
INDEX: 12
TYPE: TextItem
TEXT: R
INDEX: 13
TYPE: SectionHeaderItem
TEXT: KDIGO 2024 CLINICAL PRACTICE GUIDELINE FOR THE EVALUATION AND MANAGEMENT OF CHRONIC KIDNEY DISEASE
INDEX: 14
TYPE: TextItem
TEXT: INTERNATIONAL
INDEX: 15
TYPE: TextItem
TEXT: kidney
INDEX: 16
TYPE: TextItem
TEXT: KIDNEY
INDEX: 17
TYPE: TextItem
TEXT: DIGO
INDEX: 18
TYPE: Section

In [26]:
for i, item in enumerate(body_items):
    text = item.text.strip()

    if text.startswith("Chapter 1."):
        print("FOUND:", i, repr(text))
        break

FOUND: 160 'Chapter 1. Evaluation of CKD'


In [27]:
for i, item in enumerate(body_items[160:190], start=160):
    print("=" * 80)
    print("INDEX:", i)
    print("TYPE:", type(item).__name__)
    print("TEXT:", item.text[:500])

INDEX: 160
TYPE: SectionHeaderItem
TEXT: Chapter 1. Evaluation of CKD
INDEX: 161
TYPE: ListItem
TEXT: Table S3. Adults and children with or without CKD, estimated GFR (eGFR) based on measurements of cystatin C (eGFRcys); creatinine (eGFRcr); cystatin C and creatinine (eGFRcr-cys) versus measured GFR (mGFR; using urinary or plasma clearance of exogenous filtration marker)
INDEX: 162
TYPE: ListItem
TEXT: Table S4. Adults and children with suspected or diagnosed CKD, native kidney biopsy versus clinical or standard diagnosis or prognosis for studies evaluating diagnostic or prognostic benefit; no comparator for studies evaluating safety
INDEX: 163
TYPE: ListItem
TEXT: Table S5. Adults and children, machine-read quantitative or semiquantitative protein or albumin urine dipstick tests versus laboratory-based methods for measuring urinary protein or albumin (e.g., 24-hour urinary sample, spot urine protein-to-creatinine ratio [PCR], or albumin-to-creatinine ratio [ACR])
INDEX: 164
TYPE: Sect

In [28]:
for i, item in enumerate(body_items):
    text = item.text.strip()

    if "1.1.1 Detection of CKD" in text:
        print("FOUND:", i)
        print("TYPE:", type(item).__name__)
        print("TEXT:", repr(text))
        break

FOUND: 1765
TYPE: SectionHeaderItem
TEXT: '1.1.1 Detection of CKD'


In [29]:
for i, item in enumerate(body_items[1755:1785], start=1755):
    print("=" * 80)
    print("INDEX:", i)
    print("TYPE:", type(item).__name__)
    print("TEXT:", item.text[:500])

INDEX: 1755
TYPE: TextItem
TEXT: Efforts should be directed at people with higher risk with GFR levels >60 ml/min per 1.73 m 2 to prevent the incidence of CKD or further reductions in GFR.
INDEX: 1756
TYPE: TextItem
TEXT: Among older adults, the findings of consistently elevated RR for older adults with CKD G3a, A1, as defined by eGFRcrcys, support the inclusion of this large group in the CKD population. These elevated RRs tell us how much more likely the outcome is compared with the reference group (eGFR 90104 ml/min per 1.73 m 2 and ACR <10 mg/g [<1 mg/ mmol]). Crucially, they do not tell us what the overall likelihood of the outcome, the absolute risk, is. The absolute risk for important CKD complications is h
INDEX: 1757
TYPE: SectionHeaderItem
TEXT: Rationale for using cystatin C containing equations for CKD staging
INDEX: 1758
TYPE: TextItem
TEXT: The rationale for using cystatin C versus SCr, or a combination of both, in eGFR equations is that creatinine, which is directly linke

In [30]:
import re

def is_recommendation(text):
    return bool(
        re.match(
            r"^(Recommendation|Practice Point)\s+\d+\.\d+\.\d+\.\d+:",
            text
        )
    )

print(is_recommendation(body_items[1766].text))
print(is_recommendation(body_items[1769].text))
print(is_recommendation(body_items[1771].text))

True
True
True


In [31]:
start = 1771

for i in range(start, start + 7):
    item = body_items[i]

    print("=" * 80)
    print("INDEX:", i)
    print("TYPE:", type(item).__name__)
    print("TEXT:", item.text.strip())

INDEX: 1771
TYPE: TextItem
TEXT: Practice Point 1.1.3.1: Proof of chronicity (duration of a minimum of 3 months) can be established by:
INDEX: 1772
TYPE: ListItem
TEXT: (i) reviewofpastmeasurements/estimationsofGFR;
INDEX: 1773
TYPE: ListItem
TEXT: (ii) review of past measurements of albuminuria or proteinuria and urine microscopic examinations;
INDEX: 1774
TYPE: ListItem
TEXT: (iii) imaging findings such as reduced kidney size and reduction in cortical thickness;
INDEX: 1775
TYPE: ListItem
TEXT: (iv) kidney pathological findings such as fibrosis and atrophy;
INDEX: 1776
TYPE: ListItem
TEXT: (v) medical history, especially conditions known to cause or contribute to CKD;
INDEX: 1777
TYPE: ListItem
TEXT: (vi) repeat measurements within and beyond the 3-month point.


In [32]:
start = 1771

first_item = body_items[start]

chunk_text = first_item.text.strip()

for i in range(start + 1, len(body_items)):
    item = body_items[i]
    item_type = type(item).__name__

    if item_type == "ListItem":
        chunk_text += "\n" + item.text.strip()
    else:
        break

print(chunk_text)

Practice Point 1.1.3.1: Proof of chronicity (duration of a minimum of 3 months) can be established by:
(i) reviewofpastmeasurements/estimationsofGFR;
(ii) review of past measurements of albuminuria or proteinuria and urine microscopic examinations;
(iii) imaging findings such as reduced kidney size and reduction in cortical thickness;
(iv) kidney pathological findings such as fibrosis and atrophy;
(v) medical history, especially conditions known to cause or contribute to CKD;
(vi) repeat measurements within and beyond the 3-month point.
Practice Point 1.1.3.2: Do not assume chronicity based upon a single abnormal level for eGFR and ACR, as the finding could be the result of a recent acute kidney injury (AKI) event or acute kidney disease (AKD).
Practice Point 1.1.3.3: Consider initiation of treatments for CKD at first presentation of decreased GFR or elevated ACR if CKD is deemed likely due to presence of other clinical indicators.


In [33]:
import re

def is_recommendation(text):
    return bool(
        re.match(
            r"^(Recommendation|Practice Point)\s+\d+\.\d+\.\d+\.\d+:",
            text.strip()
        )
    )

start = 1771

chunk_items = [body_items[start]]

for i in range(start + 1, len(body_items)):
    item = body_items[i]
    text = item.text.strip()

    if is_recommendation(text):
        break

    if type(item).__name__ == "ListItem":
        chunk_items.append(item)
    else:
        break

for item in chunk_items:
    print(type(item).__name__, "→", item.text.strip())

TextItem → Practice Point 1.1.3.1: Proof of chronicity (duration of a minimum of 3 months) can be established by:
ListItem → (i) reviewofpastmeasurements/estimationsofGFR;
ListItem → (ii) review of past measurements of albuminuria or proteinuria and urine microscopic examinations;
ListItem → (iii) imaging findings such as reduced kidney size and reduction in cortical thickness;
ListItem → (iv) kidney pathological findings such as fibrosis and atrophy;
ListItem → (v) medical history, especially conditions known to cause or contribute to CKD;
ListItem → (vi) repeat measurements within and beyond the 3-month point.


In [35]:
chunk = {
    "chunk_id": "1.1.3.1",
    "type": "practice_point",
    "chapter": "Chapter 1: Evaluation of CKD",
    "section": "1.1.3 Evaluation of chronicity",
    "text": "\n".join(item.text.strip() for item in chunk_items),
    "page_start": chunk_items[0].prov[0].page_no if chunk_items[0].prov else None,
    "page_end": chunk_items[-1].prov[0].page_no if chunk_items[-1].prov else None,
}

for key, value in chunk.items():
    print(f"{key}:")
    print(value)
    print()

chunk_id:
1.1.3.1

type:
practice_point

chapter:
Chapter 1: Evaluation of CKD

section:
1.1.3 Evaluation of chronicity

text:
Practice Point 1.1.3.1: Proof of chronicity (duration of a minimum of 3 months) can be established by:
(i) reviewofpastmeasurements/estimationsofGFR;
(ii) review of past measurements of albuminuria or proteinuria and urine microscopic examinations;
(iii) imaging findings such as reduced kidney size and reduction in cortical thickness;
(iv) kidney pathological findings such as fibrosis and atrophy;
(v) medical history, especially conditions known to cause or contribute to CKD;
(vi) repeat measurements within and beyond the 3-month point.

page_start:
34

page_end:
34



In [37]:
current_chapter = None
current_section = None

for item in body_items[1763:1782]:
    text = item.text.strip()
    item_type = type(item).__name__

    if item_type == "SectionHeaderItem":
        if text.startswith("Chapter "):
            current_chapter = text
        elif re.match(r"^\d+\.\d+", text):
            current_section = text

    if is_recommendation(text):
        print("=" * 80)
        print("CHAPTER:", current_chapter)
        print("SECTION:", current_section)
        print("ITEM:", text)


CHAPTER: Chapter 1: Evaluation of CKD
SECTION: 1.1.1 Detection of CKD
ITEM: Practice Point 1.1.1.1: Test people at risk for and with chronic kidney disease (CKD) using both urine albumin measurement and assessment of glomerular filtration rate (GFR).
CHAPTER: Chapter 1: Evaluation of CKD
SECTION: 1.1.1 Detection of CKD
ITEM: Practice Point 1.1.1.2: Following incidental detection of elevated urinary albumin-to-creatinine ratio (ACR), hematuria, or low estimated GFR (eGFR), repeat tests to confirm presence of CKD.
CHAPTER: Chapter 1: Evaluation of CKD
SECTION: 1.1.2 Methods for staging of CKD
ITEM: Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtration rate (eGFRcr). If cystatin C is available, the GFR category should be estimated from the combination of creatinine and cystatin C (creatinine and cystatin C-based estimated glomerular filtration rate [eGFRcr-cys]) (1B).
CHAPTER: Chapter 1: Evaluation of CKD
SECTION: 1.1.3 Eval

In [38]:
import re

def is_recommendation(text):
    return bool(
        re.match(
            r"^(Recommendation|Practice Point)\s+\d+\.\d+\.\d+\.\d+:",
            text.strip()
        )
    )


chunks = []

current_chapter = None
current_section = None
current_chunk = None


def save_current_chunk():
    if current_chunk is not None:
        chunks.append(current_chunk)


for item in body_items:
    text = item.text.strip()
    item_type = type(item).__name__

    if not text:
        continue

    # -------------------------
    # Track chapter
    # -------------------------
    if item_type == "SectionHeaderItem":
        if text.startswith("Chapter "):
            current_chapter = text
            current_section = None
            continue

        # Track numbered clinical sections
        if re.match(r"^\d+\.\d+", text):
            current_section = text
            continue

    # -------------------------
    # Start new recommendation / practice point
    # -------------------------
    if is_recommendation(text):

        save_current_chunk()

        if text.startswith("Recommendation"):
            chunk_type = "recommendation"
        else:
            chunk_type = "practice_point"

        chunk_id = re.match(
            r"^(?:Recommendation|Practice Point)\s+(\d+\.\d+\.\d+\.\d+):",
            text
        ).group(1)

        current_chunk = {
            "chunk_id": chunk_id,
            "type": chunk_type,
            "chapter": current_chapter,
            "section": current_section,
            "text": text,
            "page_start": (
                item.prov[0].page_no
                if item.prov
                else None
            ),
            "page_end": (
                item.prov[0].page_no
                if item.prov
                else None
            ),
        }

        continue

    # -------------------------
    # Attach supporting list items
    # -------------------------
    if item_type == "ListItem" and current_chunk is not None:

        current_chunk["text"] += "\n" + text

        if item.prov:
            current_chunk["page_end"] = item.prov[0].page_no


# Save final chunk
save_current_chunk()


print(f"Created {len(chunks)} semantic chunks.")

Created 141 semantic chunks.


In [39]:
print("FIRST 3 CHUNKS")
for chunk in chunks[:3]:
    print("=" * 80)
    print("ID:", chunk["chunk_id"])
    print("TYPE:", chunk["type"])
    print("CHAPTER:", chunk["chapter"])
    print("SECTION:", chunk["section"])
    print("PAGE:", chunk["page_start"], "→", chunk["page_end"])
    print("TEXT:", chunk["text"][:800])

print("\nLAST 3 CHUNKS")
for chunk in chunks[-3:]:
    print("=" * 80)
    print("ID:", chunk["chunk_id"])
    print("TYPE:", chunk["type"])
    print("CHAPTER:", chunk["chapter"])
    print("SECTION:", chunk["section"])
    print("PAGE:", chunk["page_start"], "→", chunk["page_end"])
    print("TEXT:", chunk["text"][:800])

FIRST 3 CHUNKS
ID: 1.1.1.1
TYPE: practice_point
CHAPTER: Chapter 1: Evaluation of CKD
SECTION: 1.1.1 Detection of CKD
PAGE: 34 → 34
TEXT: Practice Point 1.1.1.1: Test people at risk for and with chronic kidney disease (CKD) using both urine albumin measurement and assessment of glomerular filtration rate (GFR).
ID: 1.1.1.2
TYPE: practice_point
CHAPTER: Chapter 1: Evaluation of CKD
SECTION: 1.1.1 Detection of CKD
PAGE: 34 → 34
TEXT: Practice Point 1.1.1.2: Following incidental detection of elevated urinary albumin-to-creatinine ratio (ACR), hematuria, or low estimated GFR (eGFR), repeat tests to confirm presence of CKD.
ID: 1.1.2.1
TYPE: recommendation
CHAPTER: Chapter 1: Evaluation of CKD
SECTION: 1.1.2 Methods for staging of CKD
PAGE: 34 → 34
TEXT: Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtration rate (eGFRcr). If cystatin C is available, the GFR category should be estimated from the combination of creatinine and cy

In [56]:
import re

def is_recommendation(text):
    return bool(
        re.match(
            r"^(Recommendation|Practice Point)\s+\d+\.\d+\.\d+\.\d+:",
            text.strip()
        )
    )

chunks = []

current_chapter = None
current_section = None
current_chunk = None


def save_current_chunk():
    if current_chunk is not None:
        current_chunk["chunk_id"] = f"chunk_{len(chunks):05d}"
        chunks.append(current_chunk)


for item in body_items:
    text = item.text.strip()
    item_type = type(item).__name__

    if not text:
        continue

    # New structural section
    if item_type == "SectionHeaderItem":
        save_current_chunk()
        current_chunk = None

        if text.startswith("Chapter "):
            current_chapter = text
            current_section = None

        elif re.match(r"^\d+\.\d+", text):
            current_section = text

        continue

    # New recommendation / practice point
    if is_recommendation(text):
        save_current_chunk()

        if text.startswith("Recommendation"):
            chunk_type = "recommendation"
        else:
            chunk_type = "practice_point"

        match = re.match(
            r"^(?:Recommendation|Practice Point)\s+"
            r"(\d+\.\d+\.\d+\.\d+):",
            text
        )

        guideline_id = match.group(1)

        current_chunk = {
            "guideline_id": guideline_id,
            "type": chunk_type,
            "chapter": current_chapter,
            "section": current_section,
            "text": text,
            "page_start": item.prov[0].page_no if item.prov else None,
            "page_end": item.prov[0].page_no if item.prov else None,
        }

        continue

    # Supporting list items belong to current recommendation
    if item_type == "ListItem" and current_chunk is not None:
        current_chunk["text"] += "\n" + text

        if item.prov:
            current_chunk["page_end"] = item.prov[0].page_no


save_current_chunk()

print("Total chunks:", len(chunks))
print("First chunk:", chunks[0])

Total chunks: 124
First chunk: {'guideline_id': '1.1.1.1', 'type': 'practice_point', 'chapter': 'Chapter 1: Evaluation of CKD', 'section': '1.1.1 Detection of CKD', 'text': 'Practice Point 1.1.1.1: Test people at risk for and with chronic kidney disease (CKD) using both urine albumin measurement and assessment of glomerular filtration rate (GFR).', 'page_start': 34, 'page_end': 34, 'chunk_id': 'chunk_00000'}


In [57]:
for chunk in chunks[-3:]:
    print("=" * 80)
    print("ID:", chunk["chunk_id"])
    print("PAGE:", chunk["page_start"], "→", chunk["page_end"])
    print("TEXT:", chunk["text"][:1000])

ID: chunk_00121
PAGE: 142 → 142
TEXT: Practice Point 5.2.2.1: Ask people with progressive CKD about uremic symptoms (e.g., reduced appetite, nausea, and level of fatigue/lethargy) at each consultation using a standardized validated assessment of uremic symptoms tool.
ID: chunk_00122
PAGE: 144 → 144
TEXT: Practice Point 5.2.3.2: Screen people with CKD G4-G5, aged >65, poor growth (pediatrics), or symptoms such as involuntary weight loss, frailty, or poor appetite twice annually for malnutrition using a validated assessment tool.
ID: chunk_00123
PAGE: 144 → 144
TEXT: Practice Point 5.2.3.3: Enable availability of appropriate medical nutrition therapy for people with signs of malnutrition, ideally under the supervision of renal dietitians or accredited nutrition providers if not available.


## 4. Create embeddings

For the first experiment we will use OpenAI embeddings.

Later we can compare:

- different embedding models
- chunk sizes
- hybrid search
- reranking
- metadata filtering

For now, keep it simple.

In [42]:
import os
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded:", bool(api_key))

client = OpenAI(api_key=api_key)

print("OpenAI client created successfully.")

API key loaded: True
OpenAI client created successfully.


In [58]:
texts = [chunk["text"] for chunk in chunks]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts,
)

embeddings = [item.embedding for item in response.data]

for chunk, embedding in zip(chunks, embeddings):
    chunk["embedding"] = embedding

print("Chunks:", len(chunks))
print("Embeddings:", len(embeddings))
print("Embedding dimensions:", len(chunks[0]["embedding"]))

Chunks: 124
Embeddings: 124
Embedding dimensions: 1536


In [44]:
texts = [chunk["text"] for chunk in chunks]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts,
)

embeddings = [item.embedding for item in response.data]

print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", len(embeddings[0]))

Number of embeddings: 124
Embedding dimensions: 1536


In [59]:
print("Chunks:", len(chunks))
print("Texts:", len(texts))
print("Embeddings:", len(embeddings))

print("Response items:", len(response.data))

Chunks: 124
Texts: 124
Embeddings: 124
Response items: 124


In [60]:
for chunk, embedding in zip(chunks, embeddings):
    chunk["embedding"] = embedding

print("Chunks:", len(chunks))
print("First chunk keys:", chunks[0].keys())
print("First embedding dimensions:", len(chunks[0]["embedding"]))

Chunks: 124
First chunk keys: dict_keys(['guideline_id', 'type', 'chapter', 'section', 'text', 'page_start', 'page_end', 'chunk_id', 'embedding'])
First embedding dimensions: 1536


In [61]:
query = """
67-year-old patient with hypertension.
Recent eGFR is 48 mL/min/1.73m2.
Creatinine is 1.6 mg/dL.
The patient reports fatigue and has reduced kidney function.
What CKD guideline recommendations may be relevant?
"""

query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=query,
)

query_embedding = query_response.data[0].embedding

print("Query embedding dimensions:", len(query_embedding))

Query embedding dimensions: 1536


In [62]:
import numpy as np

chunk_vectors = np.array([chunk["embedding"] for chunk in chunks])
query_vector = np.array(query_embedding)

similarities = (
    chunk_vectors @ query_vector
    / (
        np.linalg.norm(chunk_vectors, axis=1)
        * np.linalg.norm(query_vector)
    )
)

top_k = 5
top_indices = np.argsort(similarities)[::-1][:top_k]

for rank, idx in enumerate(top_indices, start=1):
    chunk = chunks[idx]

    print("=" * 80)
    print(f"RANK: {rank}")
    print(f"SIMILARITY: {similarities[idx]:.4f}")
    print(f"ID: {chunk['chunk_id']}")
    print(f"TYPE: {chunk['type']}")
    print(f"SECTION: {chunk['section']}")
    print(f"PAGE: {chunk['page_start']} → {chunk['page_end']}")
    print(chunk["text"][:1200])

RANK: 1
SIMILARITY: 0.6688
ID: chunk_00002
TYPE: recommendation
SECTION: 1.1.2 Methods for staging of CKD
PAGE: 34 → 34
Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtration rate (eGFRcr). If cystatin C is available, the GFR category should be estimated from the combination of creatinine and cystatin C (creatinine and cystatin C-based estimated glomerular filtration rate [eGFRcr-cys]) (1B).
RANK: 2
SIMILARITY: 0.6688
ID: chunk_00069
TYPE: recommendation
SECTION: 1.1.2 Methods for staging of CKD
PAGE: 56 → 56
Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtration rate (eGFRcr). If cystatin C is available, the GFR category should be estimated from the combination of creatinine and cystatin C (creatinine and cystatin C-based estimated glomerular filtration rate [eGFRcr-cys]) (1B).
RANK: 3
SIMILARITY: 0.6368
ID: chunk_00108
TYPE: practice_point
SECTION: 3.11.5 Di

In [63]:
# Dense similarity between all guideline chunks
chunk_vectors = np.array([chunk["embedding"] for chunk in chunks])

chunk_norms = np.linalg.norm(chunk_vectors, axis=1, keepdims=True)
normalized_vectors = chunk_vectors / chunk_norms

chunk_similarity_matrix = normalized_vectors @ normalized_vectors.T

print("Matrix shape:", chunk_similarity_matrix.shape)
print("Number of chunks:", len(chunks))

Matrix shape: (124, 124)
Number of chunks: 124


In [64]:
lambda_param = 0.7
mmr_k = 5

selected_indices = []
remaining_indices = list(top_indices)

for _ in range(mmr_k):
    if not selected_indices:
        best_idx = max(
            remaining_indices,
            key=lambda idx: similarities[idx]
        )
    else:
        best_idx = max(
            remaining_indices,
            key=lambda idx: (
                lambda_param * similarities[idx]
                - (1 - lambda_param)
                * max(
                    chunk_similarity_matrix[idx, selected_idx]
                    for selected_idx in selected_indices
                )
            )
        )

    selected_indices.append(best_idx)
    remaining_indices.remove(best_idx)

print("MMR selected chunks:")

for rank, idx in enumerate(selected_indices, start=1):
    chunk = chunks[idx]

    print("=" * 80)
    print(f"RANK: {rank}")
    print(f"SIMILARITY: {similarities[idx]:.4f}")
    print(f"ID: {chunk['chunk_id']}")
    print(f"TYPE: {chunk['type']}")
    print(f"SECTION: {chunk['section']}")
    print(f"PAGE: {chunk['page_start']} → {chunk['page_end']}")
    print(chunk["text"][:1000])

MMR selected chunks:
RANK: 1
SIMILARITY: 0.6688
ID: chunk_00002
TYPE: recommendation
SECTION: 1.1.2 Methods for staging of CKD
PAGE: 34 → 34
Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtration rate (eGFRcr). If cystatin C is available, the GFR category should be estimated from the combination of creatinine and cystatin C (creatinine and cystatin C-based estimated glomerular filtration rate [eGFRcr-cys]) (1B).
RANK: 2
SIMILARITY: 0.6368
ID: chunk_00108
TYPE: practice_point
SECTION: 3.11.5 Dietary considerations
PAGE: 111 → 111
Practice Point 3.11.5.1: Implement an individualized approach in people with CKD G3-G5 and emergent hyperkalemia that includes dietary and pharmacologic interventions and takes into consideration associated comorbidities and quality of life (QoL). Assessment and education through a renal dietitian or an accredited nutrition provider are advised.
RANK: 3
SIMILARITY: 0.6353
ID: chunk_00111
TYPE: reco

In [66]:
from collections import defaultdict

canonical_groups = defaultdict(list)

for chunk in chunks:
    key = (
        chunk["guideline_id"],
        chunk["type"],
        chunk["text"],
    )
    canonical_groups[key].append(chunk)

group_sizes = [len(group) for group in canonical_groups.values()]

print("Original chunks:", len(chunks))
print("Canonical groups:", len(canonical_groups))
print("Groups with duplicates:", sum(size > 1 for size in group_sizes))
print("Maximum occurrences of one group:", max(group_sizes))

print("\nExample duplicate groups:")

shown = 0

for key, group in canonical_groups.items():
    if len(group) > 1:
        print("-" * 80)
        print("Guideline ID:", key[0])
        print("Type:", key[1])
        print("Occurrences:", len(group))
        print("Pages:", [chunk["page_start"] for chunk in group])

        shown += 1
        if shown == 5:
            break

Original chunks: 124
Canonical groups: 87
Groups with duplicates: 37
Maximum occurrences of one group: 2

Example duplicate groups:
--------------------------------------------------------------------------------
Guideline ID: 1.1.1.1
Type: practice_point
Occurrences: 2
Pages: [34, 54]
--------------------------------------------------------------------------------
Guideline ID: 1.1.1.2
Type: practice_point
Occurrences: 2
Pages: [34, 55]
--------------------------------------------------------------------------------
Guideline ID: 1.1.2.1
Type: recommendation
Occurrences: 2
Pages: [34, 56]
--------------------------------------------------------------------------------
Guideline ID: 1.1.3.2
Type: practice_point
Occurrences: 2
Pages: [34, 57]
--------------------------------------------------------------------------------
Guideline ID: 1.1.3.3
Type: practice_point
Occurrences: 2
Pages: [34, 57]


In [68]:
for key, group in canonical_groups.items():
    if len(group) > 1:
        texts = [chunk["text"] for chunk in group]

        print("=" * 100)
        print("GUIDELINE ID:", key[0])
        print("TYPE:", key[1])
        print("TEXTS IDENTICAL:", len(set(texts)) == 1)

        for i, text in enumerate(texts, start=1):
            print(f"\n--- OCCURRENCE {i} ---")
            print(text)

GUIDELINE ID: 1.1.1.1
TYPE: practice_point
TEXTS IDENTICAL: True

--- OCCURRENCE 1 ---
Practice Point 1.1.1.1: Test people at risk for and with chronic kidney disease (CKD) using both urine albumin measurement and assessment of glomerular filtration rate (GFR).

--- OCCURRENCE 2 ---
Practice Point 1.1.1.1: Test people at risk for and with chronic kidney disease (CKD) using both urine albumin measurement and assessment of glomerular filtration rate (GFR).
GUIDELINE ID: 1.1.1.2
TYPE: practice_point
TEXTS IDENTICAL: True

--- OCCURRENCE 1 ---
Practice Point 1.1.1.2: Following incidental detection of elevated urinary albumin-to-creatinine ratio (ACR), hematuria, or low estimated GFR (eGFR), repeat tests to confirm presence of CKD.

--- OCCURRENCE 2 ---
Practice Point 1.1.1.2: Following incidental detection of elevated urinary albumin-to-creatinine ratio (ACR), hematuria, or low estimated GFR (eGFR), repeat tests to confirm presence of CKD.
GUIDELINE ID: 1.1.2.1
TYPE: recommendation
TEXTS I

In [70]:
canonical_chunks = []

for group in canonical_groups.values():
    first = group[0]

    canonical_chunk = {
        "chunk_id": first["chunk_id"],
        "guideline_id": first["guideline_id"],
        "type": first["type"],
        "chapter": first["chapter"],
        "section": first["section"],
        "text": first["text"],
        "occurrences": [
            {
                "page_start": chunk["page_start"],
                "page_end": chunk["page_end"],
            }
            for chunk in group
        ],
        "embedding": first["embedding"],
    }

    canonical_chunks.append(canonical_chunk)

print("Canonical chunks:", len(canonical_chunks))

# Check one known repeated recommendation
for chunk in canonical_chunks:
    if chunk["guideline_id"] == "1.2.2.1":
        print("\nExample:")
        print("Chunk ID:", chunk["chunk_id"])
        print("Guideline ID:", chunk["guideline_id"])
        print("Type:", chunk["type"])
        print("Occurrences:", chunk["occurrences"])
        print("Text:", chunk["text"])
        break

Canonical chunks: 87

Example:
Chunk ID: chunk_00010
Guideline ID: 1.2.2.1
Type: practice_point
Occurrences: [{'page_start': 36, 'page_end': 36}, {'page_start': 61, 'page_end': 61}]
Text: Practice Point 1.2.2.1: Use serum creatinine (SCr) and an estimating equation for initial assessment of GFR (Figure 11).


In [71]:
query_vector = np.array(query_embedding)

canonical_vectors = np.array(
    [chunk["embedding"] for chunk in canonical_chunks]
)

canonical_norms = np.linalg.norm(
    canonical_vectors,
    axis=1
)

query_norm = np.linalg.norm(query_vector)

canonical_similarities = (
    canonical_vectors @ query_vector
    / (canonical_norms * query_norm)
)

canonical_top_k = 5

canonical_top_indices = np.argsort(
    canonical_similarities
)[::-1][:canonical_top_k]

for rank, idx in enumerate(canonical_top_indices, start=1):
    chunk = canonical_chunks[idx]

    print("=" * 80)
    print(f"RANK: {rank}")
    print(f"SIMILARITY: {canonical_similarities[idx]:.4f}")
    print(f"CHUNK ID: {chunk['chunk_id']}")
    print(f"GUIDELINE ID: {chunk['guideline_id']}")
    print(f"TYPE: {chunk['type']}")
    print(f"SECTION: {chunk['section']}")
    print(f"OCCURRENCES: {chunk['occurrences']}")
    print(chunk["text"][:1000])

RANK: 1
SIMILARITY: 0.6688
CHUNK ID: chunk_00002
GUIDELINE ID: 1.1.2.1
TYPE: recommendation
SECTION: 1.1.2 Methods for staging of CKD
OCCURRENCES: [{'page_start': 34, 'page_end': 34}, {'page_start': 56, 'page_end': 56}]
Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtration rate (eGFRcr). If cystatin C is available, the GFR category should be estimated from the combination of creatinine and cystatin C (creatinine and cystatin C-based estimated glomerular filtration rate [eGFRcr-cys]) (1B).
RANK: 2
SIMILARITY: 0.6367
CHUNK ID: chunk_00048
GUIDELINE ID: 3.11.5.1
TYPE: practice_point
SECTION: 3.11.5 Dietary considerations
OCCURRENCES: [{'page_start': 46, 'page_end': 46}, {'page_start': 111, 'page_end': 111}]
Practice Point 3.11.5.1: Implement an individualized approach in people with CKD G3-G5 and emergent hyperkalemia that includes dietary and pharmacologic interventions and takes into consideration associated comorbidities 

In [74]:
# MMR for deiversity, test later

canonical_vectors = np.array(
    [chunk["embedding"] for chunk in canonical_chunks]
)

canonical_norms = np.linalg.norm(
    canonical_vectors,
    axis=1,
    keepdims=True
)

canonical_normalized_vectors = (
    canonical_vectors / canonical_norms
)

canonical_similarity_matrix = (
    canonical_normalized_vectors
    @ canonical_normalized_vectors.T
)

print(
    "Canonical similarity matrix:",
    canonical_similarity_matrix.shape
)

Canonical similarity matrix: (87, 87)


In [75]:
import numpy as np

threshold = 0.95

pairs = []

for i in range(len(canonical_chunks)):
    for j in range(i + 1, len(canonical_chunks)):
        similarity = canonical_similarity_matrix[i, j]

        if similarity >= threshold:
            pairs.append(
                {
                    "similarity": float(similarity),
                    "chunk_a": canonical_chunks[i]["chunk_id"],
                    "chunk_b": canonical_chunks[j]["chunk_id"],
                    "guideline_a": canonical_chunks[i]["guideline_id"],
                    "guideline_b": canonical_chunks[j]["guideline_id"],
                }
            )

pairs = sorted(
    pairs,
    key=lambda x: x["similarity"],
    reverse=True,
)

print("Highly similar pairs:", len(pairs))

for pair in pairs[:20]:
    print(pair)

Highly similar pairs: 9
{'similarity': 0.9998453930052968, 'chunk_a': 'chunk_00028', 'chunk_b': 'chunk_00087', 'guideline_a': '1.3.1.2', 'guideline_b': '1.3.1.2'}
{'similarity': 0.9994774585103591, 'chunk_a': 'chunk_00024', 'chunk_b': 'chunk_00084', 'guideline_a': '1.2.4.1', 'guideline_b': '1.2.4.1'}
{'similarity': 0.9989937383230585, 'chunk_a': 'chunk_00003', 'chunk_b': 'chunk_00070', 'guideline_a': '1.1.3.1', 'guideline_b': '1.1.3.1'}
{'similarity': 0.998957926498555, 'chunk_a': 'chunk_00046', 'chunk_b': 'chunk_00106', 'guideline_a': '3.11.1.1', 'guideline_b': '3.11.1.1'}
{'similarity': 0.9988225557233388, 'chunk_a': 'chunk_00033', 'chunk_b': 'chunk_00092', 'guideline_a': '3.2.2.1', 'guideline_b': '3.2.2.1'}
{'similarity': 0.9965895328607333, 'chunk_a': 'chunk_00055', 'chunk_b': 'chunk_00114', 'guideline_a': '3.15.2.1', 'guideline_b': '3.15.2.1'}
{'similarity': 0.9915532328630006, 'chunk_a': 'chunk_00027', 'chunk_b': 'chunk_00086', 'guideline_a': '1.3.1.1', 'guideline_b': '1.3.1.1'}


In [76]:
for pair in pairs:
    chunk_a = next(
        c for c in canonical_chunks
        if c["chunk_id"] == pair["chunk_a"]
    )

    chunk_b = next(
        c for c in canonical_chunks
        if c["chunk_id"] == pair["chunk_b"]
    )

    print("=" * 100)
    print(f"Similarity: {pair['similarity']:.6f}")
    print(f"A: {chunk_a['chunk_id']} | {chunk_a['guideline_id']}")
    print(chunk_a["text"])
    print()
    print(f"B: {chunk_b['chunk_id']} | {chunk_b['guideline_id']}")
    print(chunk_b["text"])

Similarity: 0.999845
A: chunk_00028 | 1.3.1.2
Practice Point 1.3.1.2: Use more accurate methods when albuminuria is detected using less accurate methods.
 Confirm reagent strip positive albuminuria and/or proteinuria by quantitative laboratory measurement and express as a ratio to urine creatinine wherever possible (i.e., quantify the ACR or PCR if initial semiquantitative tests are positive).
 Confirm ACR ‡30mg/g (‡3mg/mmol) on a random untimed urine with a subsequent first morning void in the morning midstream urine sample.

B: chunk_00087 | 1.3.1.2
Practice Point 1.3.1.2: Use more accurate methods when albuminuria is detected using less accurate methods.
 Confirm reagent strip positive albuminuria and/or proteinuria by quantitative laboratory measurement and express as a ratio to urine creatinine wherever possible (i.e., quantify the ACR or PCR if initial semiquantitative tests are positive).
 Confirm ACR ‡30 mg/g (‡3 mg/mmol) on a random untimed urine with a subsequent first mo

In [78]:
import re


def normalize_text(text):
    text = text.lower()

    # Normalize common PDF whitespace issues
    text = re.sub(r"\s+", " ", text)

    # Restore spaces around punctuation
    text = re.sub(r"\s*([,.;:()])\s*", r"\1", text)

    # Normalize common unit spacing
    text = re.sub(r"(\d)\s+(mg|g|mmol|ml)\b", r"\1\2", text)

    # Normalize hyphenation artifacts
    text = re.sub(r"(\w)-\s+(\w)", r"\1-\2", text)

    # Normalize accidental missing spaces after punctuation
    text = re.sub(r"([,.;:)])(?=[A-Za-z])", r"\1 ", text)

    return text.strip()


for pair in pairs[:3]:
    chunk_a = next(
        c for c in canonical_chunks
        if c["chunk_id"] == pair["chunk_a"]
    )

    chunk_b = next(
        c for c in canonical_chunks
        if c["chunk_id"] == pair["chunk_b"]
    )

    normalized_a = normalize_text(chunk_a["text"])
    normalized_b = normalize_text(chunk_b["text"])

    print("=" * 100)
    print(chunk_a["guideline_id"])
    print("A:", normalized_a)
    print("B:", normalized_b)
    print("IDENTICAL:", normalized_a == normalized_b)

1.3.1.2
A: practice point 1.3.1.2: use more accurate methods when albuminuria is detected using less accurate methods. confirm reagent strip positive albuminuria and/or proteinuria by quantitative laboratory measurement and express as a ratio to urine creatinine wherever possible(i. e., quantify the acr or pcr if initial semiquantitative tests are positive). confirm acr ‡30mg/g(‡3mg/mmol) on a random untimed urine with a subsequent first morning void in the morning midstream urine sample.
B: practice point 1.3.1.2: use more accurate methods when albuminuria is detected using less accurate methods. confirm reagent strip positive albuminuria and/or proteinuria by quantitative laboratory measurement and express as a ratio to urine creatinine wherever possible(i. e., quantify the acr or pcr if initial semiquantitative tests are positive). confirm acr ‡30mg/g(‡3mg/mmol) on a random untimed urine with a subsequent first morning void in the morning midstream urine sample.
IDENTICAL: True


In [79]:
for pair in pairs:
    same_guideline = (
        pair["guideline_a"] == pair["guideline_b"]
    )

    print(
        pair["guideline_a"],
        pair["guideline_b"],
        f"similarity={pair['similarity']:.6f}",
        "SAME_GUIDELINE=",
        same_guideline,
    )

1.3.1.2 1.3.1.2 similarity=0.999845 SAME_GUIDELINE= True
1.2.4.1 1.2.4.1 similarity=0.999477 SAME_GUIDELINE= True
1.1.3.1 1.1.3.1 similarity=0.998994 SAME_GUIDELINE= True
3.11.1.1 3.11.1.1 similarity=0.998958 SAME_GUIDELINE= True
3.2.2.1 3.2.2.1 similarity=0.998823 SAME_GUIDELINE= True
3.15.2.1 3.15.2.1 similarity=0.996590 SAME_GUIDELINE= True
1.3.1.1 1.3.1.1 similarity=0.991553 SAME_GUIDELINE= True
3.3.1.3 3.3.1.3 similarity=0.991460 SAME_GUIDELINE= True
3.3.2.1 3.3.2.1 similarity=0.989355 SAME_GUIDELINE= True


In [80]:
from collections import defaultdict
import numpy as np

# Start with exact grouping using guideline ID + normalized text
groups = defaultdict(list)

for chunk in chunks:
    key = (
        chunk["guideline_id"],
        normalize_text(chunk["text"]),
    )
    groups[key].append(chunk)


# Build canonical chunks
canonical_chunks_v2 = []

for group in groups.values():
    first = group[0]

    canonical_chunk = {
        "chunk_id": f"canonical_{len(canonical_chunks_v2):05d}",
        "guideline_id": first["guideline_id"],
        "type": first["type"],
        "chapter": first["chapter"],
        "section": first["section"],
        "text": first["text"],
        "occurrences": [],
        "embedding": first["embedding"],
    }

    for chunk in group:
        canonical_chunk["occurrences"].append(
            {
                "page_start": chunk["page_start"],
                "page_end": chunk["page_end"],
            }
        )

    canonical_chunks_v2.append(canonical_chunk)


print("Original chunks:", len(chunks))
print("Canonical chunks v2:", len(canonical_chunks_v2))

Original chunks: 124
Canonical chunks v2: 86


In [81]:
print("Canonical chunks:", len(canonical_chunks_v2))

# 1. Her canonical chunk'ın occurrence'ı var mı?
missing_occurrences = [
    chunk
    for chunk in canonical_chunks_v2
    if not chunk["occurrences"]
]

# 2. Gerekli alanlar mevcut mu?
required_fields = [
    "chunk_id",
    "guideline_id",
    "type",
    "chapter",
    "section",
    "text",
    "occurrences",
    "embedding",
]

missing_fields = []

for chunk in canonical_chunks_v2:
    missing = [
        field
        for field in required_fields
        if field not in chunk
    ]

    if missing:
        missing_fields.append(
            (chunk["chunk_id"], missing)
        )


print("Missing occurrences:", len(missing_occurrences))
print("Chunks with missing fields:", len(missing_fields))

# 3. Embedding boyutlarını kontrol et
embedding_dimensions = set(
    len(chunk["embedding"])
    for chunk in canonical_chunks_v2
)

print("Embedding dimensions:", embedding_dimensions)

# 4. Occurrence sayılarının dağılımı
occurrence_counts = [
    len(chunk["occurrences"])
    for chunk in canonical_chunks_v2
]

print("Total occurrences:", sum(occurrence_counts))
print(
    "Chunks with multiple occurrences:",
    sum(count > 1 for count in occurrence_counts)
)

Canonical chunks: 86
Missing occurrences: 0
Chunks with missing fields: 0
Embedding dimensions: {1536}
Total occurrences: 124
Chunks with multiple occurrences: 38


In [82]:
canonical_vectors = np.array(
    [chunk["embedding"] for chunk in canonical_chunks_v2]
)

canonical_norms = np.linalg.norm(
    canonical_vectors,
    axis=1
)

query_vector = np.array(query_embedding)
query_norm = np.linalg.norm(query_vector)

similarities = (
    canonical_vectors @ query_vector
    / (canonical_norms * query_norm)
)

top_k = 10

top_indices = np.argsort(similarities)[::-1][:top_k]

for rank, idx in enumerate(top_indices, start=1):
    chunk = canonical_chunks_v2[idx]

    print("=" * 100)
    print(f"Rank: {rank}")
    print(f"Similarity: {similarities[idx]:.4f}")
    print(f"Chunk ID: {chunk['chunk_id']}")
    print(f"Guideline ID: {chunk['guideline_id']}")
    print(f"Type: {chunk['type']}")
    print(f"Section: {chunk['section']}")
    print(f"Occurrences: {chunk['occurrences']}")
    print(f"Text: {chunk['text']}")

Rank: 1
Similarity: 0.6688
Chunk ID: canonical_00002
Guideline ID: 1.1.2.1
Type: recommendation
Section: 1.1.2 Methods for staging of CKD
Occurrences: [{'page_start': 34, 'page_end': 34}, {'page_start': 56, 'page_end': 56}]
Text: Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtration rate (eGFRcr). If cystatin C is available, the GFR category should be estimated from the combination of creatinine and cystatin C (creatinine and cystatin C-based estimated glomerular filtration rate [eGFRcr-cys]) (1B).
Rank: 2
Similarity: 0.6367
Chunk ID: canonical_00048
Guideline ID: 3.11.5.1
Type: practice_point
Section: 3.11.5 Dietary considerations
Occurrences: [{'page_start': 46, 'page_end': 46}, {'page_start': 111, 'page_end': 111}]
Text: Practice Point 3.11.5.1: Implement an individualized approach in people with CKD G3-G5 and emergent hyperkalemia that includes dietary and pharmacologic interventions and takes into consideration assoc

In [85]:
for rank, idx in enumerate(top_indices, start=1):
    chunk = canonical_chunks_v2[idx]

    print(
        rank,
        "|",
        chunk["guideline_id"],
        "|",
        f"{similarities[idx]:.4f}",
        "|",
        chunk["section"],
    )

1 | 1.1.2.1 | 0.6688 | 1.1.2 Methods for staging of CKD
2 | 3.11.5.1 | 0.6367 | 3.11.5 Dietary considerations
3 | 3.15.1.2 | 0.6353 | 3.15.1 Lipid management
4 | 3.3.1.1 | 0.6286 | 3.3.1 Protein intake
5 | 1.1.1.1 | 0.6236 | 1.1.1 Detection of CKD
6 | 3.3.1.2 | 0.6212 | 3.3.1 Protein intake
7 | 3.15.1.1 | 0.6182 | 3.15.1 Lipid management
8 | 5.2.2.1 | 0.6163 | 5.2.2 Identification and assessment of symptoms
9 | 1.1.1.2 | 0.6162 | 1.1.1 Detection of CKD
10 | 1.2.3.5 | 0.6156 | 1.2.3 Guidance to clinical laboratories


In [84]:
# structured patient data dor retrieval instead of str

In [86]:
patient = {
    "age": 67,
    "conditions": ["hypertension", "CKD"],
    "egfr": 48,
    "creatinine": 1.6,
    "symptoms": ["fatigue"],
}

print(patient)

{'age': 67, 'conditions': ['hypertension', 'CKD'], 'egfr': 48, 'creatinine': 1.6, 'symptoms': ['fatigue']}


In [ ]:
#  query decomp, initial test (to be handed over to llm)

In [92]:
query_subqueries = [
    "CKD staging and management in an adult patient with eGFR 48 mL/min/1.73m2",
    "CKD recommendations for an adult patient with hypertension",
    "CKD recommendations relevant to fatigue in an adult patient",
]

for i, subquery in enumerate(query_subqueries, start=1):
    print(f"{i}. {subquery}")

1. CKD staging and management in an adult patient with eGFR 48 mL/min/1.73m2
2. CKD recommendations for an adult patient with hypertension
3. CKD recommendations relevant to fatigue in an adult patient


In [94]:
#  bm25 index building for hybrid retrieval

In [95]:
from rank_bm25 import BM25Okapi

bm25_corpus = [
    chunk["text"].lower().split()
    for chunk in canonical_chunks_v2
]

bm25 = BM25Okapi(bm25_corpus)

print("BM25 documents:", len(bm25_corpus))

BM25 documents: 86


In [103]:
subquery = query_subqueries[0]

# BM25 scores
bm25_scores = bm25.get_scores(subquery.lower().split())

bm25_top_indices = set(
    np.argsort(bm25_scores)[::-1][:20]
)

# Dense scores
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=subquery,
)

subquery_embedding = np.array(response.data[0].embedding)

dense_vectors = np.array(
    [chunk["embedding"] for chunk in canonical_chunks_v2]
)

dense_norms = np.linalg.norm(dense_vectors, axis=1)
query_norm = np.linalg.norm(subquery_embedding)

dense_scores = (
    dense_vectors @ subquery_embedding
    / (dense_norms * query_norm)
)

dense_top_indices = set(
    np.argsort(dense_scores)[::-1][:20]
)

candidate_indices = sorted(
    bm25_top_indices | dense_top_indices
)

print("BM25 candidates:", len(bm25_top_indices))
print("Dense candidates:", len(dense_top_indices))
print("Union candidates:", len(candidate_indices))

BM25 candidates: 20
Dense candidates: 20
Union candidates: 34


In [100]:
#  test cross encoder

In [101]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3"
)

print("Reranker loaded successfully.")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker loaded successfully.


In [104]:
candidate_texts = [
    canonical_chunks_v2[idx]["text"]
    for idx in candidate_indices
]

pairs = [
    (subquery, text)
    for text in candidate_texts
]

reranker_scores = reranker.predict(pairs)

reranked_order = np.argsort(reranker_scores)[::-1]

print(f"QUERY: {subquery}\n")
print("=" * 100)

for rank, order_idx in enumerate(reranked_order[:10], start=1):
    idx = candidate_indices[order_idx]
    chunk = canonical_chunks_v2[idx]

    print(
        f"{rank} | "
        f"reranker={reranker_scores[order_idx]:.4f} | "
        f"BM25={bm25_scores[idx]:.2f} | "
        f"dense={dense_scores[idx]:.4f} | "
        f"{chunk['guideline_id']} | "
        f"{chunk['type']} | "
        f"{chunk['section']}"
    )
    print(chunk["text"])
    print("-" * 100)

QUERY: CKD staging and management in an adult patient with eGFR 48 mL/min/1.73m2

1 | reranker=0.8457 | BM25=5.06 | dense=0.5512 | 4.4.1.2 | practice_point | 4.4.1 Radiocontrast: intra-arterial and intravenous dye studies
Practice Point 4.4.1.2: The intravenous administration of radiocontrast media can be managed in accordance with consensus statements from the radiology societies in people with AKI or GFR <60 ml/min per 1.73 m 2 (CKD G3a-G5) undergoing elective investigation.
 Use of low-osmolality contrast media and iso-osmolarity contrast media
 Use of minimum radiocontrast dose to achieve a diagnostic study
 Withdrawal of nonessential potentially nephrotoxic medications (e.g., NSAIDs, diuretics, aminoglycosides, amphotericin, platins, zoledronate, and methotrexate) in people with AKI or eGFR <30 ml/min per 1.73 m 2 for 24-48 hours before and 48 hours after radiocontrast exposure
 InpeoplewitheGFR>30ml/minper1.73m 2 andwithout evidence of AKI, metformin need not be stopped befor

define structured patient state

In [105]:
patient = {
    "age": 67,
    "conditions": ["hypertension", "CKD"],
    "egfr": 48,
    "creatinine": 1.6,
    "symptoms": ["fatigue"],
}

print(patient)

{'age': 67, 'conditions': ['hypertension', 'CKD'], 'egfr': 48, 'creatinine': 1.6, 'symptoms': ['fatigue']}


In [106]:
import re


def extract_constraints(text):
    text = text.lower()

    constraints = {
        "age_min": None,
        "age_max": None,
        "egfr_min": None,
        "egfr_max": None,
        "population": "general",
    }

    # Population
    if any(
        word in text
        for word in ["children", "child", "adolescent", "adolescents", "pediatric"]
    ):
        constraints["population"] = "pediatric"
    elif "adult" in text or "adults" in text:
        constraints["population"] = "adult"

    # Age constraints
    age_min = re.search(r"(?:aged|age)\s*(?:≥|>=|>)\s*(\d+)", text)
    if age_min:
        constraints["age_min"] = int(age_min.group(1))

    age_max = re.search(r"(?:aged|age)\s*(?:≤|<=|<)\s*(\d+)", text)
    if age_max:
        constraints["age_max"] = int(age_max.group(1))

    # eGFR constraints
    egfr_min = re.search(r"egfr\s*(?:≥|>=|>)\s*(\d+)", text)
    if egfr_min:
        constraints["egfr_min"] = float(egfr_min.group(1))

    egfr_max = re.search(r"egfr\s*(?:≤|<=|<)\s*(\d+)", text)
    if egfr_max:
        constraints["egfr_max"] = float(egfr_max.group(1))

    return constraints


# Test on the top reranked candidates
for rank, order_idx in enumerate(reranked_order[:10], start=1):
    idx = candidate_indices[order_idx]
    chunk = canonical_chunks_v2[idx]

    constraints = extract_constraints(chunk["text"])

    print(
        rank,
        chunk["guideline_id"],
        constraints
    )

1 4.4.1.2 {'age_min': None, 'age_max': None, 'egfr_min': 30.0, 'egfr_max': 30.0, 'population': 'general'}
2 3.15.1.2 {'age_min': None, 'age_max': None, 'egfr_min': None, 'egfr_max': None, 'population': 'adult'}
3 4.4.1.2 {'age_min': None, 'age_max': None, 'egfr_min': None, 'egfr_max': None, 'population': 'general'}
4 1.1.2.1 {'age_min': None, 'age_max': None, 'egfr_min': None, 'egfr_max': None, 'population': 'adult'}
5 3.15.1.1 {'age_min': None, 'age_max': None, 'egfr_min': None, 'egfr_max': 60.0, 'population': 'adult'}
6 3.3.1.2 {'age_min': None, 'age_max': None, 'egfr_min': None, 'egfr_max': None, 'population': 'adult'}
7 3.15.1.3 {'age_min': None, 'age_max': None, 'egfr_min': None, 'egfr_max': None, 'population': 'adult'}
8 4.4.2.1 {'age_min': None, 'age_max': None, 'egfr_min': None, 'egfr_max': None, 'population': 'general'}
9 3.15.3.1 {'age_min': None, 'age_max': None, 'egfr_min': None, 'egfr_max': None, 'population': 'general'}
10 1.1.3.3 {'age_min': None, 'age_max': None, 'egfr_

In [107]:
from pydantic import BaseModel
from typing import Literal


class Constraint(BaseModel):
    field: Literal["age", "egfr", "condition", "population"]
    operator: Literal["<", "<=", "=", ">=", ">"]
    value: str


class Applicability(BaseModel):
    population: Literal["adult", "pediatric", "general"]
    constraints: list[Constraint]


chunk = next(
    c for c in canonical_chunks_v2
    if c["guideline_id"] == "3.15.1.2"
    and c["type"] == "recommendation"
)

print(chunk["text"])

Recommendation 3.15.1.2: In adults aged ‡50 years with CKD and eGFR ‡60 ml/min per 1.73 m 2 (GFR categories G1-G2), we recommend treatment with a statin (1B).


In [108]:
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "Extract applicability constraints from the guideline text. "
                "Only extract constraints explicitly stated in the text. "
                "Do not infer missing constraints."
            ),
        },
        {
            "role": "user",
            "content": chunk["text"],
        },
    ],
    response_format=Applicability,
)

applicability = response.choices[0].message.parsed

print(applicability.model_dump_json(indent=2))

{
  "population": "adult",
  "constraints": [
    {
      "field": "age",
      "operator": ">=",
      "value": "50"
    },
    {
      "field": "egfr",
      "operator": ">=",
      "value": "60"
    },
    {
      "field": "condition",
      "operator": "=",
      "value": "CKD"
    }
  ]
}


In [109]:
target_chunk = next(
    c for c in canonical_chunks_v2
    if c["guideline_id"] == "4.4.1.2"
)

print(target_chunk["text"])

Practice Point 4.4.1.2: The intravenous administration of radiocontrast media can be managed in accordance with consensus statements from the radiology societies in people with AKI or GFR <60 ml/min per 1.73 m 2 (CKD G3a-G5) undergoing elective investigation.


In [110]:
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "Extract applicability constraints from the guideline text. "
                "Only extract constraints explicitly stated in the text. "
                "Do not infer missing constraints. "
                "Preserve any AND/OR relationships as accurately as possible."
            ),
        },
        {
            "role": "user",
            "content": target_chunk["text"],
        },
    ],
    response_format=Applicability,
)

applicability_2 = response.choices[0].message.parsed

print(applicability_2.model_dump_json(indent=2))

{
  "population": "general",
  "constraints": [
    {
      "field": "condition",
      "operator": "=",
      "value": "AKI"
    },
    {
      "field": "egfr",
      "operator": "<",
      "value": "60 ml/min per 1.73 m2"
    }
  ]
}


In [111]:
import re
from collections import Counter

patterns = {
    "age": r"\b(?:aged|age)\b",
    "egfr_gfr": r"\b(?:eGFR|GFR)\b",
    "sex": r"\b(?:men|women|male|female)\b",
    "pregnancy": r"\b(?:pregnan|pregnancy)\w*\b",
    "pediatric": r"\b(?:child|children|adolescent|pediatric)\w*\b",
    "dialysis": r"\bdialysis\b",
    "transplant": r"\btransplant\w*\b",
    "diabetes": r"\bdiabet\w*\b",
    "hypertension": r"\bhypertension\b",
    "aki": r"\bAKI\b",
    "medication": r"\b(?:statin|ACE inhibitor|ARB|SGLT2|metformin)\w*\b",
    "numeric": r"(?:≥|≤|>=|<=|>|<)\s*\d",
    "temporal": r"\b(?:within|after|before|during|recent|newly|progressive)\b",
    "elective": r"\belective\b",
}

counts = Counter()

for chunk in canonical_chunks_v2:
    text = chunk["text"]

    for name, pattern in patterns.items():
        if re.search(pattern, text, flags=re.IGNORECASE):
            counts[name] += 1

print("Canonical chunks:", len(canonical_chunks_v2))
print()

for name, count in counts.most_common():
    print(f"{name:15} {count:3}")

Canonical chunks: 86

egfr_gfr         20
pediatric        13
numeric          11
temporal          9
age               8
medication        6
aki               4
dialysis          3
elective          2
transplant        2
diabetes          1


In [112]:
candidate_chunks = []

for chunk in canonical_chunks_v2:
    text = chunk["text"]

    matched_patterns = [
        name
        for name, pattern in patterns.items()
        if re.search(pattern, text, flags=re.IGNORECASE)
    ]

    if matched_patterns:
        candidate_chunks.append(
            {
                "chunk_id": chunk["chunk_id"],
                "guideline_id": chunk["guideline_id"],
                "type": chunk["type"],
                "matched_patterns": matched_patterns,
                "text": chunk["text"],
            }
        )

for item in candidate_chunks[:15]:
    print("=" * 100)
    print(
        item["chunk_id"],
        "|",
        item["guideline_id"],
        "|",
        item["type"],
    )
    print("Patterns:", item["matched_patterns"])
    print(item["text"])

canonical_00000 | 1.1.1.1 | practice_point
Patterns: ['egfr_gfr']
Practice Point 1.1.1.1: Test people at risk for and with chronic kidney disease (CKD) using both urine albumin measurement and assessment of glomerular filtration rate (GFR).
canonical_00001 | 1.1.1.2 | practice_point
Patterns: ['egfr_gfr']
Practice Point 1.1.1.2: Following incidental detection of elevated urinary albumin-to-creatinine ratio (ACR), hematuria, or low estimated GFR (eGFR), repeat tests to confirm presence of CKD.
canonical_00002 | 1.1.2.1 | recommendation
Patterns: ['egfr_gfr']
Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtration rate (eGFRcr). If cystatin C is available, the GFR category should be estimated from the combination of creatinine and cystatin C (creatinine and cystatin C-based estimated glomerular filtration rate [eGFRcr-cys]) (1B).
canonical_00003 | 1.1.3.1 | practice_point
Patterns: ['temporal']
Practice Point 1.1.3.1: Proof o

In [113]:
test_ids = ["1.2.3.5", "1.1.3.2", "1.1.2.1"]

test_chunks = [
    c for c in canonical_chunks_v2
    if c["guideline_id"] in test_ids
]

for c in test_chunks:
    print("=" * 100)
    print(c["guideline_id"], "|", c["type"])
    print(c["text"])

1.1.2.1 | recommendation
Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtration rate (eGFRcr). If cystatin C is available, the GFR category should be estimated from the combination of creatinine and cystatin C (creatinine and cystatin C-based estimated glomerular filtration rate [eGFRcr-cys]) (1B).
1.1.3.2 | practice_point
Practice Point 1.1.3.2: Do not assume chronicity based upon a single abnormal level for eGFR and ACR, as the finding could be the result of a recent acute kidney injury (AKI) event or acute kidney disease (AKD).
1.2.3.5 | practice_point
Practice Point 1.2.3.5: An eGFRcr level <90ml/min per 1.73m 2 can be flagged as 'low' in children and adolescents over the age of 2 years.


In [114]:
from pydantic import BaseModel
from typing import Literal


class GuidelineType(BaseModel):
    category: Literal[
        "eligibility",
        "diagnostic_interpretation",
        "treatment",
        "monitoring",
        "measurement_method",
        "general_background",
    ]
    has_patient_level_constraints: bool
    rationale: str


for c in test_chunks:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Classify the guideline text by its primary function. "
                    "Determine whether it contains patient-level conditions "
                    "that could be used to determine applicability or filtering. "
                    "Do not extract the conditions yet."
                ),
            },
            {
                "role": "user",
                "content": c["text"],
            },
        ],
        response_format=GuidelineType,
    )

    result = response.choices[0].message.parsed

    print("=" * 80)
    print(c["guideline_id"])
    print(result.model_dump_json(indent=2))

1.1.2.1
{
  "category": "diagnostic_interpretation",
  "has_patient_level_constraints": true,
  "rationale": "This recommendation targets adults at risk for chronic kidney disease (CKD) and specifies the use of creatinine-based eGFR and cystatin C for better estimation of kidney function."
}
1.1.3.2
{
  "category": "diagnostic_interpretation",
  "has_patient_level_constraints": false,
  "rationale": "This guideline advises caution in interpreting eGFR and ACR levels, implying that a patient's recent medical history should be considered when diagnosing chronic kidney conditions."
}
1.2.3.5
{
  "category": "diagnostic_interpretation",
  "has_patient_level_constraints": true,
  "rationale": "This guideline targets a specific patient population (children and adolescents over the age of 2 years) and provides a threshold for eGFRcr values that classifies them as 'low', indicating that it is relevant for assessing renal function in this age group."
}


In [115]:
# patient guideline applicability matching

In [119]:
from pydantic import BaseModel


class GuidelineMetadata(BaseModel):
    population: list[str]
    age_constraints: list[str]
    clinical_conditions: list[str]
    numeric_constraints: list[str]


response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "Extract structured metadata from the guideline text. "
                "Do not determine whether the guideline applies to a patient. "
                "Do not compare against any patient. "
                "Only extract information explicitly stated in the guideline. "
                "Preserve the semantic meaning of each constraint. "
                "Keep population or demographic requirements separate from "
                "numeric clinical thresholds. "
                "For example, 'children and adolescents over the age of 2 years' "
                "should preserve both the population information and the age "
                "constraint. "
                "Do not invent constraints that are not present."
            ),
        },
        {
            "role": "user",
            "content": guideline_for_test["text"],
        },
    ],
    response_format=GuidelineMetadata,
)

metadata = response.choices[0].message.parsed

print(metadata.model_dump_json(indent=2))

{
  "population": [
    "children and adolescents"
  ],
  "age_constraints": [
    "over the age of 2 years"
  ],
  "clinical_conditions": [],
  "numeric_constraints": [
    "eGFRcr level <90 ml/min per 1.73m2"
  ]
}


In [120]:
test_metadata_ids = [
    "1.1.2.1",  # adult + CKD + eGFR
    "1.2.3.5",  # pediatric + age + eGFR
    "4.4.1.2",  # AKI OR GFR + elective context
    "3.15.1.1", # age + CKD + eGFR + treatment
    "5.2.2.1",  # symptom / fatigue
]

test_chunks = [
    c for c in canonical_chunks_v2
    if c["guideline_id"] in test_metadata_ids
]

print("Selected chunks:", len(test_chunks))

for c in test_chunks:
    print("\n---", c["guideline_id"], "---")
    print(c["text"])

Selected chunks: 7

--- 1.1.2.1 ---
Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtration rate (eGFRcr). If cystatin C is available, the GFR category should be estimated from the combination of creatinine and cystatin C (creatinine and cystatin C-based estimated glomerular filtration rate [eGFRcr-cys]) (1B).

--- 1.2.3.5 ---
Practice Point 1.2.3.5: An eGFRcr level <90ml/min per 1.73m 2 can be flagged as 'low' in children and adolescents over the age of 2 years.

--- 3.15.1.1 ---
Practice Point 3.15.1.1: Estimate 10-year cardiovascular risk using a validated risk tool.

--- 4.4.1.2 ---
Practice Point 4.4.1.2: The intravenous administration of radiocontrast media can be managed in accordance with consensus statements from the radiology societies in people with AKI or GFR <60 ml/min per 1.73 m 2 (CKD G3a-G5) undergoing elective investigation.

--- 5.2.2.1 ---
Practice Point 5.2.2.1: Ask people with progressive CKD about urem

In [122]:
from pydantic import BaseModel, Field


class GuidelineMetadata(BaseModel):
    population: list[str] = Field(
        description=(
            "Population groups explicitly targeted or mentioned in the guideline, "
            "such as adults, children, adolescents, pregnant people, or people with CKD."
        )
    )

    age_constraints: list[str] = Field(
        description=(
            "Age-related constraints explicitly stated in the guideline, "
            "preserving their original semantic meaning, such as "
            "'age >= 50 years' or 'over the age of 2 years'."
        )
    )


metadata_results = []

for c in test_chunks:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract structured metadata from the guideline text. "
                    "Do not determine whether the guideline applies to a patient. "
                    "Do not compare against any patient. "
                    "Only extract information explicitly stated in the guideline. "
                    "Preserve the semantic meaning of each constraint. "
                    "Keep population or demographic requirements separate from "
                    "numeric clinical thresholds. "
                    "For example, 'children and adolescents over the age of 2 years' "
                    "should preserve both the population information and the age "
                    "constraint. "
                    "Do not invent constraints that are not present."
                ),
            },
            {
                "role": "user",
                "content": c["text"],
            },
        ],
        response_format=GuidelineMetadata,
    )

    metadata_results.append({
        "guideline_id": c["guideline_id"],
        "type": c["type"],
        "text": c["text"],
        "metadata": response.choices[0].message.parsed.model_dump(),
    })

for result in metadata_results:
    print("\n---", result["guideline_id"], "|", result["type"], "---")
    print(result["metadata"])


--- 1.1.2.1 | recommendation ---
{'population': ['adults at risk for CKD'], 'age_constraints': []}

--- 1.2.3.5 | practice_point ---
{'population': ['children', 'adolescents'], 'age_constraints': ['over the age of 2 years']}

--- 3.15.1.1 | practice_point ---
{'population': [], 'age_constraints': []}

--- 4.4.1.2 | practice_point ---
{'population': ['people with AKI', 'people with CKD'], 'age_constraints': ['GFR <60 ml/min per 1.73 m2']}

--- 5.2.2.1 | practice_point ---
{'population': ['people with progressive CKD'], 'age_constraints': []}

--- 3.15.1.1 | recommendation ---
{'population': ['adults'], 'age_constraints': ['aged ≥50 years']}

--- 4.4.1.2 | practice_point ---
{'population': ['people with AKI', 'people with CKD G3a-G5', 'people with eGFR <30 ml/min per 1.73 m 2', 'people with eGFR >30 ml/min per 1.73 m 2', 'people at risk for CA-AKI', 'people not undergoing dialysis', 'people receiving intravenous contrast'], 'age_constraints': []}


In [ ]:
#  add initial metadata fields to all chunks

In [123]:
from pydantic import BaseModel, Field


class GuidelineMetadata(BaseModel):
    population: list[str] = Field(
        description=(
            "Population groups explicitly targeted or mentioned in the guideline, "
            "such as adults, children, adolescents, pregnant people, or people with CKD."
        )
    )

    age_constraints: list[str] = Field(
        description=(
            "Age-related constraints explicitly stated in the guideline, "
            "preserving their original semantic meaning, such as "
            "'age >= 50 years' or 'over the age of 2 years'."
        )
    )


for i, chunk in enumerate(canonical_chunks_v2):

    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract structured metadata from the guideline text. "
                    "Only extract information explicitly stated in the text. "
                    "Do not determine whether the guideline applies to a patient. "
                    "Do not infer unstated information. "
                    "Keep population information separate from age constraints. "
                    "Preserve the semantic meaning of age constraints."
                ),
            },
            {
                "role": "user",
                "content": chunk["text"],
            },
        ],
        response_format=GuidelineMetadata,
    )

    chunk["metadata"] = response.choices[0].message.parsed.model_dump()

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(canonical_chunks_v2)}")

Processed 10/86
Processed 20/86
Processed 30/86
Processed 40/86
Processed 50/86
Processed 60/86
Processed 70/86
Processed 80/86


In [124]:
print("Total chunks:", len(canonical_chunks_v2))

metadata_count = sum(
    "metadata" in chunk
    for chunk in canonical_chunks_v2
)

print("Chunks with metadata:", metadata_count)

population_nonempty = sum(
    bool(chunk["metadata"]["population"])
    for chunk in canonical_chunks_v2
)

age_nonempty = sum(
    bool(chunk["metadata"]["age_constraints"])
    for chunk in canonical_chunks_v2
)

print("Population non-empty:", population_nonempty)
print("Age constraints non-empty:", age_nonempty)

Total chunks: 86
Chunks with metadata: 86
Population non-empty: 57
Age constraints non-empty: 13


In [125]:
for chunk in canonical_chunks_v2:
    metadata = chunk["metadata"]

    if metadata["population"] or metadata["age_constraints"]:
        print("=" * 80)
        print("Chunk:", chunk["chunk_id"])
        print("Guideline:", chunk["guideline_id"])
        print("Text:", chunk["text"])
        print("Population:", metadata["population"])
        print("Age:", metadata["age_constraints"])

Chunk: canonical_00000
Guideline: 1.1.1.1
Text: Practice Point 1.1.1.1: Test people at risk for and with chronic kidney disease (CKD) using both urine albumin measurement and assessment of glomerular filtration rate (GFR).
Population: ['people at risk for chronic kidney disease (CKD)', 'people with chronic kidney disease (CKD)']
Age: []
Chunk: canonical_00001
Guideline: 1.1.1.2
Text: Practice Point 1.1.1.2: Following incidental detection of elevated urinary albumin-to-creatinine ratio (ACR), hematuria, or low estimated GFR (eGFR), repeat tests to confirm presence of CKD.
Population: ['people with CKD']
Age: []
Chunk: canonical_00002
Guideline: 1.1.2.1
Text: Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtration rate (eGFRcr). If cystatin C is available, the GFR category should be estimated from the combination of creatinine and cystatin C (creatinine and cystatin C-based estimated glomerular filtration rate [eGFRcr-cys]) (

## 6. Define the structured clinical finding

The LLM is not allowed to return free-form prose as its primary output.

It must populate a typed schema.

The important design choice is:

`insufficient_evidence`

is a valid answer.

The model should not be forced to produce a care gap when the evidence is inadequate.

In [16]:
class Evidence(BaseModel):
    source_type: Literal["patient_record", "guideline"]
    source: str
    date: str | None = None
    page: int | None = None
    excerpt: str


class ClinicalFinding(BaseModel):
    status: Literal[
        "potential_gap",
        "no_gap",
        "insufficient_evidence"
    ]

    finding: str

    explanation: str

    patient_evidence: list[Evidence]

    guideline_evidence: list[Evidence]

    uncertainty: str

## 7. Clinical reasoning

The model receives ONLY:

- the synthetic patient record
- the retrieved guideline evidence

It should not rely on general medical knowledge to invent recommendations.

Every important conclusion should be supported by one or more pieces of evidence.

In [18]:
SYSTEM_PROMPT = """
You are a clinical evidence analysis assistant.

Your task is to compare a synthetic patient record with retrieved
clinical guideline evidence.

Important rules:

1. Use only the patient information and guideline evidence provided.
2. Do not invent patient facts.
3. Do not invent guideline recommendations.
4. Do not provide unsupported clinical recommendations.
5. If the evidence is insufficient, return "insufficient_evidence".
6. Every important conclusion must be supported by evidence.
7. Preserve provenance.
8. The output is for research and clinician review only.
"""

In [19]:
USER_PROMPT = f"""
PATIENT RECORD
==============

{patient_context}


RETRIEVED GUIDELINE EVIDENCE
============================

{guideline_context}


TASK
====

Determine whether the retrieved guideline evidence identifies a
potentially relevant care gap for this patient.

Return:

- status
- finding
- explanation
- patient evidence
- guideline evidence
- uncertainty

Do not make a clinical recommendation that is not directly supported
by the retrieved guideline evidence.
"""

In [20]:
MODEL = "gpt-4.1"

response = client.responses.parse(
    model=MODEL,
    input=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": USER_PROMPT
        }
    ],
    text_format=ClinicalFinding
)

finding = response.output_parsed

finding

ClinicalFinding(status='potential_gap', finding='Potential indication for nephrology referral in a patient with CKD (eGFR <60 mL/min/1.73m2) and hypertension suboptimally controlled on current treatment.', explanation="The patient is a 67-year-old male with hypertension and an eGFR of 48 mL/min/1.73m2, indicating CKD stage G3a or G3b. Clinical note documents that blood pressure remains elevated despite amlodipine 10 mg daily, suggesting suboptimal BP control. KDIGO 2024 guidelines note that CKD and hypertension refractory to treatment (defined as needing ≥4 antihypertensive agents) are indications for nephrology referral, but also recommend nephrology involvement (threshold >2% risk, not calculated here) and mention that decision thresholds for maximizing or optimizing therapy should be considered in people with reduced eGFR. However, the patient is currently on only one agent. There may be an unaddressed care gap in either escalating antihypertensive therapy, evaluating for secondary 

In [21]:
print(finding.model_dump_json(indent=2))

{
  "status": "potential_gap",
  "finding": "Potential indication for nephrology referral in a patient with CKD (eGFR <60 mL/min/1.73m2) and hypertension suboptimally controlled on current treatment.",
  "explanation": "The patient is a 67-year-old male with hypertension and an eGFR of 48 mL/min/1.73m2, indicating CKD stage G3a or G3b. Clinical note documents that blood pressure remains elevated despite amlodipine 10 mg daily, suggesting suboptimal BP control. KDIGO 2024 guidelines note that CKD and hypertension refractory to treatment (defined as needing ≥4 antihypertensive agents) are indications for nephrology referral, but also recommend nephrology involvement (threshold >2% risk, not calculated here) and mention that decision thresholds for maximizing or optimizing therapy should be considered in people with reduced eGFR. However, the patient is currently on only one agent. There may be an unaddressed care gap in either escalating antihypertensive therapy, evaluating for secondary

## 8. Provenance validation

Before displaying the answer to a user, we should check that the evidence
actually exists in the source material.

This is intentionally simple in v1.

Later we can make this much stronger.

In [22]:
def validate_patient_evidence(finding, patient):
    patient_text = json.dumps(patient).lower()

    results = []

    for evidence in finding.patient_evidence:
        excerpt = evidence.excerpt.lower()

        results.append({
            "excerpt": evidence.excerpt,
            "found_in_patient_record": excerpt in patient_text
        })

    return pd.DataFrame(results)


patient_validation = validate_patient_evidence(
    finding,
    patients[0]
)

patient_validation

,excerpt,found_in_patient_record
0,Patient reports increasing fatigue over recent...,True
1,eGFR: 48 mL/min/1.73m2; Creatinine: 1.6 mg/dL,False
2,Amlodipine 10 mg daily,True


In [23]:
def validate_guideline_evidence(finding, chunks):
    results = []

    for evidence in finding.guideline_evidence:

        matches = [
            chunk for chunk in chunks
            if (
                evidence.page == chunk["page"]
                and evidence.excerpt.lower() in chunk["text"].lower()
            )
        ]

        results.append({
            "page": evidence.page,
            "excerpt": evidence.excerpt,
            "found_in_guideline": len(matches) > 0
        })

    return pd.DataFrame(results)


guideline_validation = validate_guideline_evidence(
    finding,
    chunks
)

guideline_validation

,page,excerpt,found_in_guideline
0,51,• CKD and hypertension refractory to treatment...,False
1,84,There is a large body of evidence to support t...,False
2,20,The threshold GFR <60 ml/min per 1.73 m2 (GFR ...,False


## 9. Run the pipeline across all synthetic patients

We now have the basic experiment.

For each patient:

1. Build patient query.
2. Retrieve guideline evidence.
3. Ask the LLM for a structured finding.
4. Store the result.

This is deliberately small.

Evaluation comes later.

In [24]:
def build_patient_query(patient):
    return f"""
Review this patient for CKD-related guideline relevance.

Patient demographics:
{json.dumps(patient["demographics"], indent=2)}

Diagnoses:
{json.dumps(patient["diagnoses"], indent=2)}

Medications:
{json.dumps(patient["medications"], indent=2)}

Laboratory results:
{json.dumps(patient["labs"], indent=2)}

Clinical notes:
{json.dumps(patient["clinical_notes"], indent=2)}
"""

In [25]:
def run_patient(patient, top_k=5):

    query = build_patient_query(patient)

    retrieved = retrieve_guideline(
        query,
        top_k=top_k
    )

    patient_context = format_patient(patient)
    guideline_context = format_guideline_context(retrieved)

    user_prompt = f"""
PATIENT RECORD
==============

{patient_context}


RETRIEVED GUIDELINE EVIDENCE
============================

{guideline_context}


TASK
====

Determine whether the retrieved guideline evidence identifies a
potentially relevant care gap for this patient.

Return:

- status
- finding
- explanation
- patient evidence
- guideline evidence
- uncertainty

Use only the supplied evidence.
Do not invent patient facts or guideline recommendations.
"""

    response = client.responses.parse(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        text_format=ClinicalFinding
    )

    return {
        "patient": patient,
        "retrieved": retrieved,
        "finding": response.output_parsed
    }

In [26]:
results = []

for patient in patients:
    print(f"Running {patient['patient_id']}...")

    result = run_patient(patient)

    results.append(result)

print("Done.")

Running P001...
Running P002...
Running P003...
Done.


In [28]:
summary = pd.DataFrame([
    {
        "patient_id": result["patient"]["patient_id"],
        "status": result["finding"].status,
        "finding": result["finding"].finding,
        "uncertainty": result["finding"].uncertainty
    }
    for result in results
])

summary

,patient_id,status,finding,uncertainty
0,P001,potential_gap,Potential under-assessment and suboptimal mana...,Uncertainty remains as the record does not sta...
1,P002,potential_gap,No documented use of a standardized validated ...,It is possible that a standardized tool was us...
2,P003,no_gap,No care gap identified regarding assessment of...,None of the guideline referral criteria or CKD...


In [29]:
def display_result(result):

    patient = result["patient"]
    finding = result["finding"]

    print("=" * 90)
    print(f"PATIENT: {patient['patient_id']}")
    print("=" * 90)

    print(f"\nSTATUS: {finding.status}")

    print("\nFINDING:")
    print(finding.finding)

    print("\nEXPLANATION:")
    print(finding.explanation)

    print("\nPATIENT EVIDENCE:")
    for evidence in finding.patient_evidence:
        print(f"- {evidence.excerpt}")
        print(f"  Source: {evidence.source}")
        print(f"  Date: {evidence.date}")

    print("\nGUIDELINE EVIDENCE:")
    for evidence in finding.guideline_evidence:
        print(f"- {evidence.excerpt}")
        print(f"  Source: {evidence.source}")
        print(f"  Page: {evidence.page}")

    print("\nUNCERTAINTY:")
    print(finding.uncertainty)

In [30]:
display_result(results[0])

PATIENT: P001

STATUS: potential_gap

FINDING:
Potential under-assessment and suboptimal management of CKD-related complications and hypertension in a patient with CKD stage G3b.

EXPLANATION:
The patient, a 67-year-old male with hypertension and reduced kidney function (eGFR 48 mL/min/1.73m2), reports increasing fatigue, a recognized uremic symptom, and has persistent hypertension despite being on Amlodipine. According to the KDIGO 2024 guideline, people with progressive CKD should be routinely assessed for uremic symptoms and have their management individualized, including medication review and consideration for referral to nephrology if blood pressure is refractory to treatment. There is no documentation of standardized assessment of symptoms, medication review, or nephrology referral, which may represent a care gap.

PATIENT EVIDENCE:
- Patient reports increasing fatigue over recent months. Blood pressure remains elevated despite current medication. Recent blood testing demonstrate

# Retrieval sensitivity experiment

Before moving to Streamlit, we should understand whether the answer changes
when we change the number of retrieved guideline chunks.

This is NOT our formal evaluation framework yet.

It is simply an experiment to understand the behaviour of the pipeline.

In [31]:
for top_k in [2, 3, 5, 8]:

    print("\n" + "=" * 90)
    print(f"TOP K = {top_k}")
    print("=" * 90)

    result = run_patient(
        patients[0],
        top_k=top_k
    )

    print("Status:", result["finding"].status)
    print("Finding:", result["finding"].finding)
    print("Uncertainty:", result["finding"].uncertainty)


TOP K = 2
Status: potential_gap
Finding: Symptoms and management considerations in a patient with CKD and refractory hypertension may not have been fully addressed according to guideline recommendations.
Uncertainty: It is unclear whether a standardized symptom assessment tool was used or if referral to nephrology was considered. The patient's current antihypertensive regimen is not at the ≥4 agent threshold, but hypertension is already uncontrolled. Guideline criteria for referral and symptom assessment require further documentation for confirmation.

TOP K = 3
Status: potential_gap
Finding: Possible care gap in comprehensive assessment and specialist referral for CKD and refractory hypertension.
Uncertainty: Incomplete data: No details on albuminuria, full medication list, or use of validated uremia assessment tools. Number of antihypertensive agents is below the strict threshold for 'refractory' hypertension by guideline definition, but BP is not controlled. No documentation that c

In [32]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

output = []

for result in results:
    output.append({
        "patient_id": result["patient"]["patient_id"],
        "finding": result["finding"].model_dump(),
        "retrieved_guideline_chunks": result["retrieved"]
    })

with open(
    output_dir / "initial_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:", output_dir / "initial_results.json")

Saved: outputs/initial_results.json


# Initial conclusions

Questions to answer before productionising:

1. Did retrieval return genuinely relevant guideline sections?
2. Were the retrieved sections sufficient to support the LLM's conclusion?
3. Did the model correctly distinguish between:
   - potential gap
   - no gap
   - insufficient evidence?
4. Did the model preserve patient evidence?
5. Did the model preserve guideline provenance?
6. Did changing `top_k` materially change the result?
7. Where did retrieval fail?
8. Where did reasoning fail?
9. What information would need to be deterministic rather than LLM-generated?

## Next iteration

Only after the basic pipeline works:

- improve chunking
- add metadata
- add deterministic clinical checks
- add better provenance validation
- add more synthetic patients
- introduce an agent only where it adds value
- build evaluation datasets
- measure latency and cost
- extract the notebook code into reusable Python modules
- build the Streamlit interface
- add FastAPI if useful

This notebook is the experiment. The production application comes later.